# CONFIG FOR PATHS

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from typing import Optional
import gzip

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────


PATHS = {
    # inputs
    "snp_deltas":          "parquet/clinvar_new_deltas.parquet",
    "indel_deltas":        "parquet/clinvar_indel_deltas.parquet",
    "variant_summary":     "variant_summary.txt.gz",
    "submission_summary":  "submission_summary.txt.gz",
    # MANE and Promoter processed
    "MANE_processed":      "metadata/MANE_processed.csv", # region annotation
    "Promoter_processed":  "metadata/Promoter_processed.csv",
    # outputs
    "snp_strict":          "parquet/snps_strict_test.parquet",
    "indel_strict":        "parquet/indels_strict_test.parquet",
    "snp_annotated":       "parquet/snps_annotated_test.parquet",
    "indel_annotated":     "parquet/indels_annotated_test.parquet",
    "indel_annotated_signaled": "parquet/annotated_indels_signaled.parquet",
    "snp_annotated_signaled": "parquet/annotated_snps_signaled.parquet",
    # trackes_metadata
    "tracks_metadata":     "metadata/functional_tracks_metadata_human.csv"
}


# MAIN PIPELINE: from model's deltas to filtered (Rationale and review status), annotated by Gencode gtf parquet



*filtering*:
  - Requires Assembly Hg38
  - Deduplicate on (chrom, pos, ref, alt) — drop ALL copies of duplicates
  - Require a real rationale (not placeholder)
  - Require qualifying review status (at least 2 golds stars)

  *notes*:
  1. Vaiant_summary use
  - snps - don't have #VariationID and GeneSymbol in delta parquet. getting it from variant_summary by matching ['chrom','pos', 'alt', 'ref']
  - indels - don't have GeneSymbol, getting it from variant_summary
  - hg38 filtrations by "Assembly" column

  2. Submission_summary use:
  - obtaining review_status and Rationale from submission_summary by matching #VariationID


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# MAIN PIPELINE DETLAS.parquet -> snps_strict.parquet -> snps_annotated.parquet
# ──────────────────────────────────────────────────────────────────────────────

"""
NT_V3 Variant Annotation Pipeline
===================================
Integrates NT_V3 model predictions (ALT vs REF delta tracks) for SNPs and indels,
enriches with ClinVar rationales and review status, and annotates with GENCODE GTF
genomic features.

Pipeline structure:
  PART 1 – SNP variants:  load deltas → map VariationID → fetch rationales → filter
  PART 2 – Indel variants: load deltas (ID pre-mapped) → fetch rationales → filter
  PART 3 – GENCODE GTF annotation for surviving variants (both SNPs and indels)
"""

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from typing import Optional
from collections import defaultdict

import gzip

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────




# ClinVar variant_summary columns needed for SNP ID mapping
CLINVAR_SUMMARY_COLS = [
    "Chromosome", "PositionVCF", "ReferenceAlleleVCF",
    "AlternateAlleleVCF", "VariationID", "Type", "GeneSymbol", "Assembly", "ReviewStatus"
]

# ClinVar submission_summary columns needed for rationale extraction
SUBMISSION_COLS = [
    "#VariationID", "Description",
    "ExplanationOfInterpretation", "ReviewStatus",
]

# Review-status constants
ALLOWED_OTHER_STATUS = {
    "no assertion criteria provided",
    "no classification provided",
}
CRITERIA_SINGLE = "criteria provided, single submitter"

# GTF feature priority (lower index = higher priority)
FEATURE_PRIORITY = [
    "start_codon", "stop_codon", "CDS",
    "UTR5", "UTR3",           # after reclassification
    "UTR",                    # fallback if reclassification fails
    "exon", "transcript", "gene",
]

REVIEW_STATUS_TO_GOLD_STARS = {
    'criteria provided, single submitter': 1,
    'criteria provided, multiple submitters, no conflicts': 2,
    'criteria provided, conflicting interpretations': 1,
    'no assertion criteria provided': np.nan,
    'reviewed by expert panel': 3,
    'no assertion provided': np.nan,
    'no interpretation for the single variant': np.nan,
    'practice guideline': 4,
    }

Z_COLS_SNP =  ['MLM_logprob_ref','MLM_logprob_alt',
                'MLM_logprob_delta']

Z_COLS_INDEL = ['MLM_logprob_ref','MLM_logprob_alt',
                'MLM_logprob_delta',
                'EMB_l2_dist','EMB_max_pos_dist','EMB_mean_pos_dist']


# ──────────────────────────────────────────────────────────────────────────────
# PART 1 & 2 HELPERS — ClinVar rationale + review-status enrichment
# ──────────────────────────────────────────────────────────────────────────────

def _clean_text(text_series: pd.Series) -> str:
    """Concatenate non-empty unique rationale strings (stable order)."""
    valid = [
        str(t) for t in text_series.dropna()
        if str(t).strip() not in ("-", "")
    ]
    seen: set[str] = set()
    uniq: list[str] = []
    for v in valid:
        if v not in seen:
            uniq.append(v)
            seen.add(v)
    return " | ".join(uniq)


def _combine_rationale(row: pd.Series) -> str:
    """Merge Description + ExplanationOfInterpretation into one text block."""
    parts = []
    if row.get("Description"):
        parts.append(f"DESC: {row['Description']}")
    if row.get("ExplanationOfInterpretation"):
        parts.append(f"EXPL: {row['ExplanationOfInterpretation']}")
    return "\n".join(parts) if parts else "No detailed rationale provided."


def aggregate_review_status(sub_df: pd.DataFrame) -> pd.Series:
    """
    Per-VariationID review-status aggregation.

    Rules:
      - Single submission with 'reviewed by expert panel' → expert_panel
      - ≥2 submissions where all are in {criteria_single} ∪ {allowed_other},
        with ≥1 criteria_single, and (≥2 criteria OR ≥1 other)
        → criteria_provided,_multiple_submitters,_no_conflicts
      - Everything else → pd.NA  (excluded downstream)
    """
    tmp = (
        sub_df[["#VariationID", "ReviewStatus"]]
        .dropna(subset=["#VariationID"])
        .copy()
    )
    tmp["#VariationID"] = tmp["#VariationID"].astype("Int64")
    tmp["ReviewStatus"] = tmp["ReviewStatus"].astype(str).str.strip().str.lower()

    allowed_all = {CRITERIA_SINGLE} | ALLOWED_OTHER_STATUS

    def _agg(s: pd.Series):
        vals = s.dropna().tolist()
        n = len(vals)
        if n == 0:
            return pd.NA
        if n == 1 and vals[0] == "reviewed by expert panel":
            return "reviewed_by_expert_panel"
        if n <= 1:
            return pd.NA
        if any(v not in allowed_all for v in vals):
            return pd.NA
        n_criteria = sum(v == CRITERIA_SINGLE for v in vals)
        if n_criteria < 1:
            return pd.NA
        n_other = n - n_criteria
        if n_criteria >= 2 or n_other >= 1:
            return "criteria_provided,_multiple_submitters,_no_conflicts"
        return pd.NA

    return tmp.groupby("#VariationID")["ReviewStatus"].apply(_agg)


def _load_submission_summary(path: str) -> pd.DataFrame:
    """Load ClinVar submission_summary (tab-separated, 18 header rows)."""
    print(f"  Loading submission summary from {path} ...")
    return pd.read_csv(
        path, sep="\t", compression="gzip",
        usecols=SUBMISSION_COLS, skiprows=18, low_memory=False,
    )


def fetch_rationales_and_filter(
    mapped_df: pd.DataFrame,
    submission_summary_path: str,
    min_gold_stars: int = 2,
    sub_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """
    Enrich *mapped_df* with ClinVar rationales + review status, then apply
    strict filtering:
      1. Deduplicate on (chrom, pos, ref, alt) — drop ALL copies of duplicates
      2. Require a real rationale (not placeholder)
      3. Require qualifying review status

    Parameters
    ----------
    mapped_df : DataFrame with at least [chrom, pos, ref, alt, #VariationID]
    submission_summary_path : path to submission_summary.txt.gz
    sub_df : pre-loaded submission summary (optional, avoids re-reading)

    Returns
    -------
    strict_df : filtered DataFrame
    """
    if sub_df is None:
        sub_df = _load_submission_summary(submission_summary_path)

    sub_df = sub_df.dropna(subset=["#VariationID"]).copy()

    # filter by gold stars
    mapped_df = mapped_df[mapped_df["gold_stars"] >= min_gold_stars].copy()

    # ── aggregate rationales ──
    print("  Aggregating rationales ...")
    grouped = (
        sub_df
        .groupby("#VariationID", as_index=False)
        .agg({
            "Description": _clean_text,
            "ExplanationOfInterpretation": _clean_text,
        })
    )
    grouped["FullRationale"] = grouped.apply(_combine_rationale, axis=1)
    grouped = grouped[["#VariationID", "FullRationale"]]

    # ── merge rationales ──
    print("  Merging rationales ...")
    final_df = mapped_df.copy()
    final_df["FullRationale"] = "No rationale provided."

    final_df["#VariationID"] = final_df["#VariationID"].astype("Int64")
    grouped["#VariationID"] = grouped["#VariationID"].astype("Int64")

    rationale_map = grouped.set_index("#VariationID")["FullRationale"]
    mask_has_id = final_df["#VariationID"].notna()
    final_df.loc[mask_has_id, "FullRationale"] = (
        final_df.loc[mask_has_id, "#VariationID"]
        .map(rationale_map)
        .fillna("No rationale provided.")
    )

    # changed review status to gold stars and filtered by gold stars


    # ── deduplicate (drop ALL copies of duplicate loci) ──
    dup_mask = final_df.duplicated(subset=["chrom", "pos", "ref", "alt"], keep=False)
    dedup_df = final_df[~dup_mask].copy()
    print(f"  After dedup: {len(dedup_df):,}  (dropped {dup_mask.sum():,} duplicate-locus rows)")

    # ── strict filter ──
    placeholder_rationales = {"No rationale provided.", "No detailed rationale provided."}



    strict_df = dedup_df[
        (~dedup_df["FullRationale"].isin(placeholder_rationales))
    ].copy()
    print(f"  After strict filter: {len(strict_df):,}")

    return strict_df





# ──────────────────────────────────────────────────────────────────────────────
# PART 1 — SNP VARIANTS
# ──────────────────────────────────────────────────────────────────────────────

def load_snp_deltas(path: str) -> pd.DataFrame:
    """Load NT_V3 SNP delta predictions (index cols + D_BED_* tracks)."""
    pf = pq.ParquetFile(path)
    all_cols = pf.schema.names
    index_cols = ["chrom", "pos", "ref", "alt", "label"]
    delta_cols = [c for c in all_cols if c.startswith("D_BED_")]
    # cols = index_cols + delta_cols
    cols = all_cols
    print(f"  Loading {len(cols)} columns ({len(delta_cols)} BED features) ...")

    df = pf.read(columns=cols).to_pandas()
    df["pos"] = pd.to_numeric(df["pos"], errors="coerce").astype("Int64")
    return df


def map_snp_variation_ids(
    delta_df: pd.DataFrame,
    variant_summary_path: str,
) -> pd.DataFrame:
    """Map ClinVar #VariationID + GeneSymbol onto SNP delta df via (chrom,pos,ref,alt)."""
    print("  Loading ClinVar variant_summary ...")
    var_df = pd.read_csv(
        variant_summary_path, sep="\t", compression="gzip",
        usecols=CLINVAR_SUMMARY_COLS, low_memory=False,
    )
    var_df.rename(columns={
        "Chromosome": "chrom",
        "PositionVCF": "pos",
        "ReferenceAlleleVCF": "ref",
        "AlternateAlleleVCF": "alt",
        "VariationID": "#VariationID",
        "ReviewStatus": "review_status",
    }, inplace=True)

    # normalise chrom to "chrN"
    var_df["chrom"] = var_df["chrom"].astype(str)
    if not var_df["chrom"].iloc[0].startswith("chr"):
        var_df["chrom"] = "chr" + var_df["chrom"]

    # GRCh38 only
    var_df = var_df[var_df["Assembly"] == "GRCh38"].copy()

    # review status to gold stars
    var_df["gold_stars"] = var_df["review_status"].map(REVIEW_STATUS_TO_GOLD_STARS)


    # SNVs only
    snp_df = var_df[var_df["Type"] == "single nucleotide variant"].copy()
    snp_df["pos"] = pd.to_numeric(snp_df["pos"], errors="coerce").astype("Int64")

    print("  Merging VariationIDs ...")
    merged = delta_df.merge(
        snp_df[["chrom", "pos", "ref", "alt", "#VariationID", "GeneSymbol", "gold_stars"]],
        on=["chrom", "pos", "ref", "alt"],
        how="left",
    )
    n_matched = merged["#VariationID"].notna().sum()
    print(f"  Matched {n_matched:,} / {len(merged):,} variants to ClinVar IDs")
    return merged

def zscore_cols(df: pd.DataFrame, z_cols) -> pd.DataFrame:
    """Z-score normalize all columns with "D_BED_" or "D_BW_" in the name."""
    for col in z_cols:
        df[col] = (df[col] - df[col].mean()) / df[col].std()
    return df


def process_snps(paths: dict, sub_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """Full SNP pipeline: load → map IDs → rationales → filter → save."""
    print("\n══ PART 1: SNP Variants ══")
    delta_df = load_snp_deltas(paths["snp_deltas"])
    z_cols = Z_COLS_SNP
    delta_df = zscore_cols(delta_df, z_cols)

    mapped_df = map_snp_variation_ids(delta_df, paths["variant_summary"])
    strict_df = fetch_rationales_and_filter(
        mapped_df, paths["submission_summary"], sub_df=sub_df,
    )
    strict_df.to_parquet(paths["snp_strict"], index=False)
    print(f"  Saved → {paths['snp_strict']}")
    return strict_df


# ──────────────────────────────────────────────────────────────────────────────
# PART 2 — INDEL VARIANTS
# ──────────────────────────────────────────────────────────────────────────────

def load_indel_deltas(path: str) -> pd.DataFrame:
    """Load NT_V3 indel delta predictions (variant_id already present)."""
    pf = pq.ParquetFile(path)
    print(f"  Loading {len(pf.schema.names)} columns ...")
    df = pf.read().to_pandas()
    df.rename(columns={"variant_id": "#VariationID"}, inplace=True)
    df["pos"] = pd.to_numeric(df["pos"], errors="coerce").astype("Int64")
    return df

def map_indels_variation_ids(
    delta_df: pd.DataFrame,
    variant_summary_path: str,
) -> pd.DataFrame:
    """Map ClinVar #VariationID + GeneSymbol onto SNP delta df via (chrom,pos,ref,alt)."""
    print("  Loading ClinVar variant_summary ...")
    var_df = pd.read_csv(
        variant_summary_path, sep="\t", compression="gzip",
        usecols=CLINVAR_SUMMARY_COLS, low_memory=False,
    )
    var_df.rename(columns={
        "Chromosome": "chrom",
        "PositionVCF": "pos",
        "ReferenceAlleleVCF": "ref",
        "AlternateAlleleVCF": "alt",
        "VariationID": "#VariationID",
        "ReviewStatus": "review_status",
    }, inplace=True)

    # normalise chrom to "chrN"
    var_df["chrom"] = var_df["chrom"].astype(str)
    if not var_df["chrom"].iloc[0].startswith("chr"):
        var_df["chrom"] = "chr" + var_df["chrom"]

    # Indels only
    indel_df = var_df[var_df["Type"].isin(["Deletion", "Insertion", "Indel"])].copy()
    indel_df["pos"] = pd.to_numeric(indel_df["pos"], errors="coerce").astype("Int64")

    # collapse to ONE row per VariationID (take first non-null GeneSymbol)
    gene_symbol_per_id = (
        indel_df
            .dropna(subset=["GeneSymbol"])
            .drop_duplicates(subset=["#VariationID"])
            [["#VariationID", "GeneSymbol", "review_status"]]
    )

    print("  Merging GeneSymbols ...")
    merged = delta_df.merge(
        gene_symbol_per_id,
        on="#VariationID",
        how="left",
    )

    # review status to gold stars
    merged["gold_stars"] = merged["review_status"].map(REVIEW_STATUS_TO_GOLD_STARS)


    n_matched = merged["GeneSymbol"].notna().sum()
    print(f"  Matched {n_matched:,} / {len(merged):,} variants to ClinVar IDs")
    return merged


def process_indels(paths: dict, sub_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """Full indel pipeline: load → rationales → filter → save."""
    print("\n══ PART 2: Indel Variants ══")
    indel_df = load_indel_deltas(paths["indel_deltas"])
    z_cols = Z_COLS_INDEL

    indel_df = zscore_cols(indel_df, z_cols)
    mapped_df = map_indels_variation_ids(indel_df, paths["variant_summary"])
    strict_df = fetch_rationales_and_filter(
        mapped_df, paths["submission_summary"], sub_df=sub_df,
    )
    strict_df.to_parquet(paths["indel_strict"], index=False)
    print(f"  Saved → {paths['indel_strict']}")
    return strict_df


# ──────────────────────────────────────────────────────────────────────────────
# PART 3 — GENCODE GTF ANNOTATION
# ──────────────────────────────────────────────────────────────────────────────

# ---------------------------------------------------------------------------
# 0.A Column definitions
# ---------------------------------------------------------------------------
annotation_columns = [
    'gene', 'mRNA', 'mRNA_promoter', 'mRNA_exon', 'coding_sequence',
    'start_codon', 'stop_codon', 'five_prime_UTR', 'three_prime_UTR',
    'mRNA_intron', 'mRNA_splice', 'lncRNA', 'lncRNA_promoter', 'lncRNA_exon',
    'snRNA', 'snRNA_promoter', 'snRNA_exon', 'antisenseRNA',
    'antisenseRNA_promoter', 'antisenseRNA_exon', 'telomeraseRNA',
    'telomeraseRNA_promoter', 'telomeraseRNA_exon', 'RNaseMRPRNA',
    'RNaseMRPRNA_promoter', 'RNaseMRPRNA_exon', 'snoRNA', 'snoRNA_promoter',
    'snoRNA_exon', 'other'
]

RNA_TYPES = ['lncRNA', 'snRNA', 'antisenseRNA', 'telomeraseRNA',
             'RNaseMRPRNA', 'snoRNA']

# ---------------------------------------------------------------------------
# 0.B PART 3 - Helpers
# ---------------------------------------------------------------------------

def collapse_region_class(region: str) -> str:
    """
    Collapse a comma-separated region annotation string into a single
    high-level genomic region class for prompting.

    Priority order (highest → lowest):
      CODING > SPLICE > UTR_5 > UTR_3 > PROMOTER > INTRONIC > GENIC_OTHER > OTHER
    """

    if not isinstance(region, str) or not region.strip():
        return "OTHER"

    parts = {r.strip() for r in region.split(",")}

    # --- Priority-based classification ---
    if {"coding_sequence", "start_codon", "stop_codon"} & parts:
        if "mRNA_splice" in parts:
            return "CODING, SPLICE"
        return "CODING"

    if "mRNA_splice" in parts:
        return "SPLICE"

    if "five_prime_UTR" in parts:
        return "UTR_5"

    if "three_prime_UTR" in parts:
        return "UTR_3"

    if "mRNA_promoter" in parts:
        return "PROMOTER"

    if "mRNA_intron" in parts:
        return "INTRONIC"

    if parts & {"gene", "mRNA", "mRNA_exon"}:
        return "GENIC_OTHER"

    return "OTHER"

# ---------------------------------------------------------------------------
# 1. One-time preprocessing — call once before annotating
# ---------------------------------------------------------------------------
def preprocess(MANE_raw, Promoter_raw):
    """
    Pre-cast types and build per-chromosome lookup structures.
    Returns (mane_by_chrom, promoter_by_chrom, mane_full_by_chrom).
    """
    MANE = MANE_raw.copy()
    Promoter = Promoter_raw.copy()

    # Cast once
    MANE['Start'] = MANE['Start'].astype(np.int64)
    MANE['End'] = MANE['End'].astype(np.int64)
    Promoter['Promoter_Start'] = Promoter['Promoter_Start'].astype(np.int64)
    Promoter['Promoter_End'] = Promoter['Promoter_End'].astype(np.int64)

    # Normalize chromosome naming — strip 'chr' prefix in MANE to match VCF style
    # Adjust this depending on your actual data
    MANE['chrom_key'] = MANE['Chromosome']
    Promoter['chrom_key'] = Promoter['Chromosome'].astype(str)

    # Group by chromosome for O(1) chrom lookup
    mane_by_chrom = {k: g for k, g in MANE.groupby('chrom_key')}
    promoter_by_chrom = {k: g for k, g in Promoter.groupby('chrom_key')}

    # For the "full MANE" lookups (transcript exons/CDS across whole chrom)
    # we also build parent-indexed lookups
    mane_parent_idx = {}
    for chrom, df in mane_by_chrom.items():
        parent_groups = {}
        for parent_val, grp in df.groupby('Parent', sort=False):
            parent_groups[parent_val] = grp
        mane_parent_idx[chrom] = parent_groups

    return mane_by_chrom, promoter_by_chrom, mane_parent_idx


# ---------------------------------------------------------------------------
# 2. Fast overlap query using sorted arrays + searchsorted
# ---------------------------------------------------------------------------
def build_interval_index(df, start_col='Start', end_col='End'):
    """
    Returns (starts, ends, indices) sorted by start for searchsorted.
    """
    starts = df[start_col].values
    ends = df[end_col].values
    order = np.argsort(starts)
    return starts[order], ends[order], df.index.values[order]


def query_overlaps(starts_sorted, ends_sorted, idx_sorted, pos):
    """
    Find all intervals containing pos.
    intervals where start <= pos AND end >= pos.
    """
    # All intervals that start <= pos
    right = np.searchsorted(starts_sorted, pos, side='right')  # first index where start > pos
    # Among those (0..right-1), filter end >= pos
    if right == 0:
        return np.array([], dtype=np.int64)
    candidate_ends = ends_sorted[:right]
    mask = candidate_ends >= pos
    return idx_sorted[:right][mask]


# ---------------------------------------------------------------------------
# 3. Per-variant annotation (operates on dicts, not pd.Series)
# ---------------------------------------------------------------------------
def annotate_variant(chrom, pos, mane_by_chrom, promoter_by_chrom, mane_parent_idx):
    """
    Annotate a single variant. Returns dict of annotation flags + transcript sets.
    """
    result = {col: 0 for col in annotation_columns}
    transcript_set = set()
    promoter_transcript_set = set()

    chrom_str = str(chrom)

    # --- Overlap with MANE ---
    mane_df = mane_by_chrom.get(chrom_str)
    if mane_df is not None and len(mane_df) > 0:
        mask = (mane_df['Start'].values <= pos) & (mane_df['End'].values >= pos)
        annotation = mane_df[mask]
    else:
        annotation = pd.DataFrame()

    # --- Overlap with Promoter ---
    prom_df = promoter_by_chrom.get(chrom_str)
    if prom_df is not None and len(prom_df) > 0:
        mask = (prom_df['Promoter_Start'].values <= pos) & (prom_df['Promoter_End'].values >= pos)
        annotation_promoter = prom_df[mask]
    else:
        annotation_promoter = pd.DataFrame()

    if annotation.empty and annotation_promoter.empty:
        result['other'] = 1
        return result, transcript_set, promoter_transcript_set

    types = set(annotation['Feature'].unique()) if not annotation.empty else set()
    types_promoter = set(annotation_promoter['Feature'].unique()) if not annotation_promoter.empty else set()

    # --- gene ---
    if 'gene' in types:
        result['gene'] = 1

    # --- mRNA ---
    if 'mRNA' in types:
        result['mRNA'] = 1
        tids = set(annotation.loc[annotation['Feature'] == 'mRNA', 'transcript_id'].dropna())
        transcript_set.update(tids)

        # Get parent-index for this chrom
        parent_idx = mane_parent_idx.get(chrom_str, {})

        for tid in tids:
            rna_key = f'rna-{tid}'

            # Strand
            id_match = annotation[annotation['ID'] == rna_key]
            if id_match.empty:
                continue
            strand = id_match['Strand'].iloc[0]

            # Exon/CDS overlapping this position
            ann_exon = annotation[(annotation['Parent'] == rna_key) & (annotation['Feature'] == 'exon')]
            ann_cds = annotation[(annotation['Parent'] == rna_key) & (annotation['Feature'] == 'CDS')]

            # Full transcript exons/CDS (from full MANE, not just overlapping)
            full_exon = parent_idx.get(rna_key)
            if full_exon is not None:
                tr_exon = full_exon[full_exon['Feature'] == 'exon']
                tr_cds = full_exon[full_exon['Feature'] == 'CDS']
            else:
                tr_exon = pd.DataFrame()
                tr_cds = pd.DataFrame()

            if not ann_cds.empty and not ann_exon.empty:
                # Exon + CDS
                result['mRNA_exon'] = 1
                result['coding_sequence'] = 1

                if not tr_cds.empty:
                    cds_starts = tr_cds['Start'].values
                    cds_ends = tr_cds['End'].values
                    if strand == '+':
                        start_1 = cds_starts.min()
                        if start_1 <= pos <= start_1 + 2:
                            result['start_codon'] = 1
                        stop_3 = cds_ends.max()
                        if stop_3 - 2 <= pos <= stop_3:
                            result['stop_codon'] = 1
                    else:
                        start_1 = cds_ends.max()
                        if start_1 - 2 <= pos <= start_1:
                            result['start_codon'] = 1
                        stop_3 = cds_starts.min()
                        if stop_3 <= pos <= stop_3 + 2:
                            result['stop_codon'] = 1

            elif ann_cds.empty and not ann_exon.empty:
                # UTR
                result['mRNA_exon'] = 1
                if not tr_exon.empty and not tr_cds.empty:
                    exon_starts = tr_exon['Start'].values
                    exon_ends = tr_exon['End'].values
                    cds_starts = tr_cds['Start'].values
                    cds_ends = tr_cds['End'].values

                    if strand == '+':
                        five_start = exon_starts.min()
                        five_end = cds_starts.min() - 1
                        if five_start <= pos <= five_end:
                            result['five_prime_UTR'] = 1
                        three_start = cds_ends.max() + 1
                        three_end = exon_ends.max()
                        if three_start <= pos <= three_end:
                            result['three_prime_UTR'] = 1
                    else:
                        five_start = exon_ends.max()
                        five_end = cds_ends.max() + 1
                        if five_end <= pos <= five_start:
                            result['five_prime_UTR'] = 1
                        three_start = cds_starts.min() - 1
                        three_end = exon_starts.min()
                        if three_end <= pos <= three_start:
                            result['three_prime_UTR'] = 1

            elif ann_cds.empty and ann_exon.empty:
                # Intron
                result['mRNA_intron'] = 1
                if not tr_exon.empty:
                    ex_starts = tr_exon['Start'].values
                    ex_ends = tr_exon['End'].values
                    # Splice: pos within 2bp of any exon boundary
                    splice_positions = np.concatenate([
                        ex_starts - 1, ex_starts - 2,
                        ex_ends + 1, ex_ends + 2
                    ])
                    if pos in splice_positions:
                        result['mRNA_splice'] = 1

    # --- mRNA promoter ---
    if 'mRNA' in types_promoter:
        result['mRNA_promoter'] = 1
        tids = set(annotation_promoter.loc[
            annotation_promoter['Feature'] == 'mRNA', 'transcript_id'
        ].dropna())
        promoter_transcript_set.update(tids)

    # --- Other RNA types ---
    for rna in RNA_TYPES:
        if rna in types:
            result[rna] = 1
            tids = set(annotation.loc[annotation['Feature'] == rna, 'transcript_id'].dropna())
            transcript_set.update(tids)
            for tid in tids:
                rna_key = f'rna-{tid}'
                ann_exon = annotation[(annotation['Parent'] == rna_key) & (annotation['Feature'] == 'exon')]
                if not ann_exon.empty:
                    result[f'{rna}_exon'] = 1

        if rna in types_promoter:
            result[f'{rna}_promoter'] = 1
            tids = set(annotation_promoter.loc[
                annotation_promoter['Feature'] == rna, 'transcript_id'
            ].dropna())
            promoter_transcript_set.update(tids)

    return result, transcript_set, promoter_transcript_set


# ---------------------------------------------------------------------------
# 4. Main vectorized driver — no multiprocessing overhead for small-medium data
# ---------------------------------------------------------------------------
def annotate_clinvar(ClinVar, MANE_raw, Promoter_raw, use_parallel=False, n_workers=None):
    """
    Annotate all ClinVar variants.

    For datasets < 100k variants, single-process with pre-indexed lookups
    is usually faster than multiprocessing (avoids serialization overhead).

    For > 100k, set use_parallel=True.
    """
    from tqdm import tqdm

    # Preprocess once
    mane_by_chrom, promoter_by_chrom, mane_parent_idx = preprocess(MANE_raw, Promoter_raw)

    chroms = ClinVar['chrom'].values
    positions = ClinVar['pos'].astype(np.int64).values

    all_results = []
    all_tsets = []
    all_ptsets = []

    if use_parallel:
        from concurrent.futures import ProcessPoolExecutor
        import functools

        # For true parallelism, partition by chromosome to avoid sharing state
        # But for simplicity, use ThreadPoolExecutor (GIL-bound but avoids pickle)
        from concurrent.futures import ThreadPoolExecutor

        if n_workers is None:
            import os
            n_workers = os.cpu_count()

        def _worker(i):
            return annotate_variant(
                chroms[i], positions[i],
                mane_by_chrom, promoter_by_chrom, mane_parent_idx
            )

        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            futures = list(tqdm(
                executor.map(_worker, range(len(chroms))),
                total=len(chroms),
                desc="Annotating variants"
            ))
            for res, tset, ptset in futures:
                all_results.append(res)
                all_tsets.append(tset)
                all_ptsets.append(ptset)
    else:
        for i in tqdm(range(len(chroms)), desc="Annotating variants"):
            res, tset, ptset = annotate_variant(
                chroms[i], positions[i],
                mane_by_chrom, promoter_by_chrom, mane_parent_idx
            )
            all_results.append(res)
            all_tsets.append(tset)
            all_ptsets.append(ptset)

    # Build result DataFrame
    ann_df = pd.DataFrame(all_results, index=ClinVar.index)
    for col in annotation_columns:
        ClinVar[col] = ann_df[col].values
    ClinVar['transcript_set'] = all_tsets
    ClinVar['promoter_transcript_set'] = all_ptsets

    ClinVar['region'] = (
    ClinVar[annotation_columns]
    .apply(lambda r: ','.join(r.index[r == 1]), axis=1)
    )


    ClinVar["region_class"] = ClinVar["region"].apply(collapse_region_class)

    return ClinVar



# ──────────────────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────────────────

def main():
    """Run the full pipeline: SNPs → Indels → GTF annotation."""

    # Pre-load submission summary once (shared between SNPs and indels)
    sub_df = _load_submission_summary(PATHS["submission_summary"])

    # ── PART 1: SNPs ──
    snp_strict = process_snps(PATHS, sub_df=sub_df)

    # ── PART 2: Indels ──
    indel_strict = process_indels(PATHS, sub_df=sub_df)

    # ── PART 3: GENCODE GTF annotation ──
    MANE = pd.read_csv(PATHS["MANE_processed"])
    Promoter = pd.read_csv(PATHS["Promoter_processed"])
    print("\n══ PART 3: GENCODE GTF Annotation ══")

    snp_annotated = annotate_clinvar(snp_strict, MANE, Promoter)
    snp_annotated.to_parquet(PATHS["snp_annotated"], index=False)
    print(f"  Saved → {PATHS['snp_annotated']}  ({len(snp_annotated):,} variants)")

    indel_annotated = annotate_clinvar(indel_strict, MANE, Promoter)
    indel_annotated.to_parquet(PATHS["indel_annotated"], index=False)
    print(f"  Saved → {PATHS['indel_annotated']}  ({len(indel_annotated):,} variants)")


    print("\n✓ Pipeline complete.")
    # return snp_annotated, indel_annotated


if __name__ == "__main__":
    main()

  Loading submission summary from submission_summary.txt.gz ...

══ PART 2: Indel Variants ══
  Loading 7270 columns ...
  Loading ClinVar variant_summary ...
  Merging GeneSymbols ...
  Matched 13,064 / 13,064 variants to ClinVar IDs
  Aggregating rationales ...
  Merging rationales ...
  Aggregating review status ...
  Variants before filtering: 13,064
  After dedup: 13,064  (dropped 0 duplicate-locus rows)
  After strict filter: 9,017
  Saved → parquet/indels_strict_test.parquet

══ PART 3: GENCODE GTF Annotation ══
  Loading GENCODE GTF ...
  Loaded 6,980,582 GTF rows across 77,078 genes
  Annotating variants with GTF features (vectorised) ...
  GTF annotation summary:
CDS            8390
transcript      487
exon            109
start_codon      14
stop_codon       13
intergenic        4
  Saved → parquet/indels_annotated_test.parquet  (9,017 variants)

✓ Pipeline complete.


# **Extracting and aggregating MLM, BED, BW signals - PRE LLM**

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# FAST VECTORIZED SIGNAL EXTRACTION FOR LLM
# ──────────────────────────────────────────────────────────────────────────────
#
# Replaces row-wise df.apply() with vectorized NumPy operations.
# Expected speedup: 50-100x on numeric aggregation, 5-10x on string formatting.
# ──────────────────────────────────────────────────────────────────────────────

"""
 Extract comprehensive signal information including:
    - BED signals (delta + reference)
    - MLM signals (all relevant columns)
    - BW signals (delta + reference + metadata annotation)
"""
import pandas as pd
import numpy as np
from typing import Optional, Dict, List, Tuple


# =============================================================================
# CORE VECTORIZED HELPERS
# =============================================================================

def _vectorized_top_k_indices(matrix: np.ndarray, k: int) -> np.ndarray:
    """
    Get top-k column indices per row by absolute value.
    Uses argpartition (O(n)) instead of full argsort (O(n log n)).

    Args:
        matrix: (n_rows, n_cols) array of values (already filled, no NaN)
        k: number of top signals to extract

    Returns:
        (n_rows, k) array of column indices, sorted by descending |value|
    """
    n_rows, n_cols = matrix.shape
    k_actual = min(k, n_cols)
    abs_matrix = np.abs(matrix)

    if k_actual >= n_cols:
        # Just argsort the whole thing
        top_indices = np.argsort(-abs_matrix, axis=1)[:, :k_actual]
        return top_indices

    # argpartition: O(n) to get top-k (unsorted)
    top_indices = np.argpartition(abs_matrix, -k_actual, axis=1)[:, -k_actual:]

    # Sort just the k elements per row (O(k log k) per row)
    rows = np.arange(n_rows)[:, None]
    top_vals = abs_matrix[rows, top_indices]
    sort_within = np.argsort(-top_vals, axis=1)
    top_indices_sorted = np.take_along_axis(top_indices, sort_within, axis=1)

    return top_indices_sorted


def _vectorized_top_k_signed(matrix: np.ndarray, k: int, direction: str) -> np.ndarray:
    """
    Get top-k column indices for gains (direction='gain') or losses (direction='loss').

    For gains: top-k by descending value where value > 0.01
    For losses: top-k by ascending value where value < -0.01

    Returns:
        (n_rows, k) array of column indices. Padded with -1 for rows with fewer signals.
    """
    n_rows, n_cols = matrix.shape
    k_actual = min(k, n_cols)

    if direction == 'gain':
        # Mask non-gains, sort descending
        work = matrix.copy()
        work[work <= 0.01] = -np.inf  # push non-gains to bottom
        top_indices = np.argsort(-work, axis=1)[:, :k_actual]
        # Mark invalid entries
        rows = np.arange(n_rows)[:, None]
        vals = matrix[rows, top_indices]
        top_indices[vals <= 0.01] = -1
    else:  # loss
        work = matrix.copy()
        work[work >= -0.01] = np.inf  # push non-losses to bottom
        top_indices = np.argsort(work, axis=1)[:, :k_actual]
        rows = np.arange(n_rows)[:, None]
        vals = matrix[rows, top_indices]
        top_indices[vals >= -0.01] = -1

    return top_indices


# =============================================================================
# STRING FORMATTING (BULK)
# =============================================================================

def _format_signal_strings(
    matrix: np.ndarray,
    ref_matrix: np.ndarray,
    top_k_indices: np.ndarray,
    feature_names: np.ndarray,
    include_ref: bool = True
) -> List[str]:
    """
    Build formatted signal strings for all rows at once.

    Format: "feature=delta(ref=X); feature2=delta2(ref=Y); ..."

    Args:
        matrix: (n_rows, n_cols) delta values
        ref_matrix: (n_rows, n_cols) reference values (may contain NaN)
        top_k_indices: (n_rows, k) indices into columns. -1 means skip.
        feature_names: (n_cols,) array of feature name strings
        include_ref: whether to include reference values

    Returns:
        List of formatted strings, one per row
    """
    n_rows, k = top_k_indices.shape
    results = []

    for i in range(n_rows):
        parts = []
        for j in range(k):
            col_idx = top_k_indices[i, j]
            if col_idx == -1:
                continue
            d = matrix[i, col_idx]
            if d == 0 and not include_ref:
                continue

            part = f"{feature_names[col_idx]}={d:.3f}"
            if include_ref:
                r = ref_matrix[i, col_idx]
                if not np.isnan(r):
                    part += f"(ref={r:.3f})"
            parts.append(part)

        results.append("; ".join(parts) if parts else "None")

    return results


def _format_bw_signal_strings(
    matrix: np.ndarray,
    ref_matrix: np.ndarray,
    top_k_indices: np.ndarray,
    descriptions: np.ndarray,
) -> List[str]:
    """
    Build formatted BW signal strings with metadata descriptions.
    Format: "tissue|biosample|assay=delta(ref=X); ..."
    """
    n_rows, k = top_k_indices.shape
    results = []

    for i in range(n_rows):
        parts = []
        for j in range(k):
            col_idx = top_k_indices[i, j]
            if col_idx == -1:
                continue
            d = matrix[i, col_idx]

            part = f"{descriptions[col_idx]}={d:.3f}"
            r = ref_matrix[i, col_idx]
            if not np.isnan(r):
                part += f"(ref={r:.3f})"
            parts.append(part)

        results.append("; ".join(parts) if parts else "None")

    return results


# =============================================================================
# METADATA HELPERS
# =============================================================================

def _build_metadata_lookup(metadata_df: Optional[pd.DataFrame]) -> Dict[str, Dict]:
    """Build file_id -> metadata dict."""
    if metadata_df is None:
        return {}

    lookup = {}
    for _, row in metadata_df.iterrows():
        file_id = row.get('file_id', '')
        lookup[file_id] = {
            'biosample_type': row.get('biosample_type', ''),
            'tissue': row.get('tissue', ''),
            'assay': row.get('assay', ''),
            'experiment_target': row.get('experiment_target', ''),
            'dataset': row.get('dataset', '')
        }
    return lookup


def _build_bw_descriptions(bw_delta_cols: List[str], metadata_lookup: Dict) -> np.ndarray:
    """Build description array for BW tracks."""
    descriptions = []
    for col in bw_delta_cols:
        track_id = col.replace('D_BW_', '')
        meta = metadata_lookup.get(track_id, {})
        if meta:
            desc = f"{meta.get('tissue', 'Unknown')}|{meta.get('biosample_type', '')}|{meta.get('assay', '')}"
            if meta.get('experiment_target'):
                desc += f"|{meta['experiment_target']}"
        else:
            desc = track_id
        descriptions.append(desc)
    return np.array(descriptions)


# =============================================================================
# MLM EXTRACTION (VECTORIZED)
# =============================================================================

def _extract_mlm_columns(df: pd.DataFrame, variant_type: str) -> Tuple[pd.DataFrame, List[str]]:
    """Extract MLM columns and return as sub-DataFrame."""
    if variant_type == 'snp':
        mlm_cols = ['REF_5mer', 'ALT_5mer', 'LLR', 'MLM_Prior', 'MLM_Delta']
    else:
        mlm_cols = [
            'LLR', 'MLM_Prior', 'MLM_Delta', 'REF_5mer', 'ALT_5mer',
            'MLM_KL_mean', 'MLM_KL_max', 'MLM_logprob_ref', 'MLM_logprob_alt',
            'MLM_logprob_delta', 'EMB_cosine_dist', 'EMB_l2_dist',
            'EMB_max_pos_dist', 'EMB_mean_pos_dist'
        ]

    existing_cols = [c for c in mlm_cols if c in df.columns]
    return df[existing_cols] if existing_cols else pd.DataFrame(index=df.index), existing_cols


def _format_mlm_summary_strings(df: pd.DataFrame, variant_type: str) -> List[str]:
    """Vectorized MLM summary string construction."""
    n = len(df)
    results = [""] * n

    # Define which columns go into the summary and their format
    core_cols = [
        ('LLR', 'LLR', '.3f'),
        ('MLM_Delta', 'MLM_Delta', '.3f'),
        ('MLM_Prior', 'MLM_Prior', '.3f'),
        ('MLM_logprob_delta', 'LogProb_Delta', '.1f'),
        ('MLM_logprob_ref', 'LogProb_Ref', '.1f'),

    ]

    indel_cols = [
        ('EMB_cosine_dist', 'EMB_cos', '.3f'),
        ('MLM_KL_max', 'KL_max', '.3f'),
        ('MLM_KL_mean', 'KL_mean', '.3f'),
    ]

    cols_to_format = core_cols + (indel_cols if variant_type == 'indel' else [])

    # Pre-extract arrays
    arrays = {}
    for col_name, _, _ in cols_to_format:
        if col_name in df.columns:
            arrays[col_name] = df[col_name].values

    for i in range(n):
        parts = []
        for col_name, label, fmt in cols_to_format:
            if col_name in arrays:
                val = arrays[col_name][i]
                if not pd.isna(val):
                    parts.append(f"{label}={val:{fmt}}")
        results[i] = "; ".join(parts) if parts else "None"

    return results


# =============================================================================
# MAIN FAST EXTRACTION
# =============================================================================

def extract_signals_comprehensive_fast(
    df: pd.DataFrame,
    delta_cols: List[str],
    metadata_df: Optional[pd.DataFrame] = None,
    k: int = 5,
    variant_type: str = 'snp'
) -> pd.DataFrame:
    """
    Vectorized replacement for row-wise extract_signals_comprehensive.
    Operates on entire DataFrame at once using NumPy.

    Args:
        df: Full variant DataFrame
        delta_cols: List of delta column names (D_BED_*, D_BW_*)
        metadata_df: Track metadata DataFrame
        k: Number of top signals per category
        variant_type: 'snp' or 'indel'

    Returns:
        DataFrame with signal columns to join back
    """
    n_rows = len(df)
    result_df = pd.DataFrame(index=df.index)

    # -----------------------------------------------------------------
    # Separate column types
    # -----------------------------------------------------------------
    bed_delta_cols = [c for c in delta_cols if c.startswith('D_BED_')]
    bw_delta_cols = [c for c in delta_cols if c.startswith('D_BW_')]

    # -----------------------------------------------------------------
    # BED SIGNALS
    # -----------------------------------------------------------------
    if bed_delta_cols:
        bed_feature_names = np.array([c.replace('D_BED_', '') for c in bed_delta_cols])
        bed_ref_cols = [c.replace('D_BED_', 'REF_BED_') for c in bed_delta_cols]
        # Ensure ref cols exist
        bed_ref_cols_exist = [c for c in bed_ref_cols if c in df.columns]

        # Extract matrices
        bed_matrix = df[bed_delta_cols].fillna(0).values.astype(np.float64)

        # Build ref matrix (align columns)
        if bed_ref_cols_exist:
            bed_ref_matrix = np.full_like(bed_matrix, np.nan)
            for i, delta_col in enumerate(bed_delta_cols):
                ref_col = delta_col.replace('D_BED_', 'REF_BED_')
                if ref_col in df.columns:
                    bed_ref_matrix[:, i] = pd.to_numeric(df[ref_col], errors='coerce').values
        else:
            bed_ref_matrix = np.full_like(bed_matrix, np.nan)

        # Aggregate metrics (fully vectorized — no Python loops)
        abs_bed = np.abs(bed_matrix)
        result_df['BED_Max_Abs_Delta'] = abs_bed.max(axis=1)
        result_df['BED_N_Strong_Signals'] = (abs_bed > 0.3).sum(axis=1)

        # Top-K indices
        bed_top_abs_idx = _vectorized_top_k_indices(bed_matrix, k)
        bed_top_gain_idx = _vectorized_top_k_signed(bed_matrix, k, 'gain')
        bed_top_loss_idx = _vectorized_top_k_signed(bed_matrix, k, 'loss')

        # Format strings
        result_df['BED_Top_Abs'] = _format_signal_strings(
            bed_matrix, bed_ref_matrix, bed_top_abs_idx, bed_feature_names
        )
        result_df['BED_Top_Gains'] = _format_signal_strings(
            bed_matrix, bed_ref_matrix, bed_top_gain_idx, bed_feature_names
        )
        result_df['BED_Top_Losses'] = _format_signal_strings(
            bed_matrix, bed_ref_matrix, bed_top_loss_idx, bed_feature_names
        )
    else:
        result_df['BED_Max_Abs_Delta'] = 0
        result_df['BED_N_Strong_Signals'] = 0
        result_df['BED_Top_Abs'] = "None"
        result_df['BED_Top_Gains'] = "None"
        result_df['BED_Top_Losses'] = "None"

    # -----------------------------------------------------------------
    # BW SIGNALS (with metadata)
    # -----------------------------------------------------------------
    if bw_delta_cols:
        metadata_lookup = _build_metadata_lookup(metadata_df)
        bw_descriptions = _build_bw_descriptions(bw_delta_cols, metadata_lookup)
        bw_ref_cols = [c.replace('D_BW_', 'REF_BW_') for c in bw_delta_cols]

        bw_matrix = df[bw_delta_cols].fillna(0).values.astype(np.float64)

        bw_ref_matrix = np.full_like(bw_matrix, np.nan)
        for i, delta_col in enumerate(bw_delta_cols):
            ref_col = delta_col.replace('D_BW_', 'REF_BW_')
            if ref_col in df.columns:
                bw_ref_matrix[:, i] = pd.to_numeric(df[ref_col], errors='coerce').values

        # Aggregate metrics
        abs_bw = np.abs(bw_matrix)
        result_df['BW_Max_Abs_Delta'] = abs_bw.max(axis=1)
        result_df['BW_N_Strong_Signals'] = (abs_bw > 0.3).sum(axis=1)

        # Top-K indices
        bw_top_abs_idx = _vectorized_top_k_indices(bw_matrix, k)
        bw_top_gain_idx = _vectorized_top_k_signed(bw_matrix, k, 'gain')
        bw_top_loss_idx = _vectorized_top_k_signed(bw_matrix, k, 'loss')

        # Format strings
        result_df['BW_Top_Abs'] = _format_bw_signal_strings(
            bw_matrix, bw_ref_matrix, bw_top_abs_idx, bw_descriptions
        )
        result_df['BW_Top_Gains'] = _format_bw_signal_strings(
            bw_matrix, bw_ref_matrix, bw_top_gain_idx, bw_descriptions
        )
        result_df['BW_Top_Losses'] = _format_bw_signal_strings(
            bw_matrix, bw_ref_matrix, bw_top_loss_idx, bw_descriptions
        )
    else:
        result_df['BW_Max_Abs_Delta'] = 0
        result_df['BW_N_Strong_Signals'] = 0
        result_df['BW_Top_Abs'] = "None"
        result_df['BW_Top_Gains'] = "None"
        result_df['BW_Top_Losses'] = "None"

    # -----------------------------------------------------------------
    # MLM SIGNALS
    # -----------------------------------------------------------------
    mlm_sub, mlm_cols_used = _extract_mlm_columns(df, variant_type)

    # Individual MLM columns
    for col in mlm_cols_used:
        result_df[f'MLM_{col}'] = mlm_sub[col].values if col in mlm_sub.columns else np.nan

    # Summary string
    result_df['MLM_Summary'] = _format_mlm_summary_strings(df, variant_type)

    return result_df


# =============================================================================
# LLM PROMPT FORMATTER (SINGLE-ROW, ON-DEMAND)
# =============================================================================

def extract_signals_for_llm(
    row,
    delta_cols: List[str],
    metadata_df: Optional[pd.DataFrame] = None,
    k: int = 5,
    variant_type: str = 'snp'
) -> Dict:
    """
    Format signals for a SINGLE variant's LLM prompt.
    Call this at inference time, NOT during batch processing.

    Can work from either:
    - Pre-computed signal columns (if extract_signals_comprehensive_fast was run)
    - Raw delta columns (slower, self-contained fallback)
    """

    # --- Check if pre-computed columns exist ---
    if 'BED_Top_Abs' in row.index and 'BW_Top_Abs' in row.index:
        # Fast path: reformat pre-computed strings into LLM-friendly text
        bed_text = _reformat_compact_to_llm(row.get('BED_Top_Abs', 'None'), 'BED')
        bw_text = _reformat_compact_to_llm(row.get('BW_Top_Abs', 'None'), 'BW')
        mlm_text = _reformat_mlm_to_llm(row, variant_type)

        return {
            'BED_Signals_Text': bed_text,
            'BW_Signals_Text': bw_text,
            'MLM_Signals_Text': mlm_text,
            'Full_Signal_Block': f"""**BED Feature Changes (Top {k}):**
{bed_text}

**Epigenomic Track Changes (Top {k}):**
{bw_text}

**MLM Sequence Context:**
{mlm_text}"""
        }

    # --- Slow fallback: compute from raw columns ---
    # (Same logic as original, for standalone use)
    return _extract_signals_for_llm_from_raw(row, delta_cols, metadata_df, k, variant_type)


def _reformat_compact_to_llm(compact_str: str, signal_type: str) -> str:
    """Convert 'feature=delta(ref=X); ...' to verbose LLM format."""
    if compact_str == "None" or not compact_str:
        return "  None"

    lines = []
    for entry in compact_str.split("; "):
        # Parse: "feature=delta(ref=X)" or "feature=delta"
        if "=" not in entry:
            continue

        name_part, rest = entry.split("=", 1)

        # Check for ref value
        if "(ref=" in rest:
            delta_str, ref_part = rest.split("(ref=", 1)
            ref_str = ref_part.rstrip(")")
            lines.append(f"  - {name_part}: Δ={float(delta_str):+.3f} (REF={float(ref_str):.3f})")
        else:
            lines.append(f"  - {name_part}: Δ={float(rest):+.3f}")

    return "\n".join(lines) if lines else "  None"


def _reformat_mlm_to_llm(row, variant_type: str) -> str:
    """Reformat MLM columns into LLM-friendly text."""
    if variant_type == 'snp':
        cols = ['REF_5mer', 'ALT_5mer', 'LLR', 'MLM_Prior', 'MLM_Delta']
    else:
        cols = [
            'LLR', 'MLM_Prior', 'MLM_Delta', 'REF_5mer', 'ALT_5mer',
            'MLM_KL_mean', 'MLM_KL_max', 'MLM_logprob_ref', 'MLM_logprob_alt',
            'MLM_logprob_delta', 'EMB_cosine_dist', 'EMB_l2_dist',
            'EMB_max_pos_dist', 'EMB_mean_pos_dist'
        ]

    lines = []
    for col in cols:
        # Check both raw and prefixed versions
        val = row.get(f'MLM_{col}', row.get(col, np.nan))
        if not pd.isna(val):
            if isinstance(val, str):
                lines.append(f"  - {col}: {val}")
            else:
                lines.append(f"  - {col}: {val:.4f}")

    return "\n".join(lines) if lines else "  None"


def _extract_signals_for_llm_from_raw(row, delta_cols, metadata_df, k, variant_type):
    """Fallback: original row-wise extraction for single-variant use."""
    bed_delta_cols = [c for c in delta_cols if c.startswith('D_BED_')]
    bw_delta_cols = [c for c in delta_cols if c.startswith('D_BW_')]

    metadata_lookup = _build_metadata_lookup(metadata_df)

    # BED
    bed_entries = []
    for col in bed_delta_cols:
        delta = row.get(col, 0)
        if pd.isna(delta): delta = 0
        ref = row.get(col.replace('D_BED_', 'REF_BED_'), np.nan)
        feature = col.replace('D_BED_', '')
        bed_entries.append({'feature': feature, 'delta': delta,
                           'ref': ref if not pd.isna(ref) else None})

    bed_sorted = sorted(bed_entries, key=lambda x: abs(x['delta']), reverse=True)[:k]
    bed_lines = []
    for e in bed_sorted:
        line = f"  - {e['feature']}: Δ={e['delta']:+.3f}"
        if e['ref'] is not None: line += f" (REF={e['ref']:.3f})"
        bed_lines.append(line)
    bed_text = "\n".join(bed_lines) if bed_lines else "  None"

    # BW
    bw_entries = []
    for col in bw_delta_cols:
        delta = row.get(col, 0)
        if pd.isna(delta): delta = 0
        ref = row.get(col.replace('D_BW_', 'REF_BW_'), np.nan)
        track_id = col.replace('D_BW_', '')
        meta = metadata_lookup.get(track_id, {})
        if meta:
            desc = f"{meta.get('tissue', '?')} / {meta.get('biosample_type', '?')} / {meta.get('assay', '?')}"
            if meta.get('experiment_target'): desc += f" / {meta['experiment_target']}"
        else:
            desc = track_id
        bw_entries.append({'description': desc, 'delta': delta,
                          'ref': ref if not pd.isna(ref) else None})

    bw_sorted = sorted(bw_entries, key=lambda x: abs(x['delta']), reverse=True)[:k]
    bw_lines = []
    for e in bw_sorted:
        line = f"  - {e['description']}: Δ={e['delta']:+.3f}"
        if e['ref'] is not None: line += f" (REF={e['ref']:.3f})"
        bw_lines.append(line)
    bw_text = "\n".join(bw_lines) if bw_lines else "  None"

    # MLM
    mlm_text = _reformat_mlm_to_llm(row, variant_type)

    return {
        'BED_Signals_Text': bed_text,
        'BW_Signals_Text': bw_text,
        'MLM_Signals_Text': mlm_text,
        'Full_Signal_Block': f"""**BED Feature Changes (Top {k}):**
{bed_text}

**Epigenomic Track Changes (Top {k}):**
{bw_text}

**MLM Sequence Context:**
{mlm_text}"""
    }


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main(variant_type_arg, PATHS, k_signals = 10):
    """
    extracting top k main signals from BED, BW tracks.
    tranlsating track ids to their annotations using metadata for tracks

    Usage:
        df = main("snp", PATHS, 10)
    """
    import pyarrow.parquet as pq

    # --- CONFIGURATION ---
    if variant_type_arg == "indel":
        INPUT_FILE = PATHS["indel_annotated"]
        METADATA_FILE = PATHS["tracks_metadata"]
        OUTPUT_FILE = PATHS["indel_annotated_signaled"]
    elif variant_type_arg == "snp":
        INPUT_FILE = PATHS["snp_annotated"]
        METADATA_FILE = PATHS["tracks_metadata"]
        OUTPUT_FILE = PATHS["snp_annotated_signaled"]
    else:
        raise ValueError(f"Unknown variant type: {variant_type_arg}")


    # --- LOAD DATA ---
    print("Loading data...")
    variant_df = pq.read_table(INPUT_FILE).to_pandas()
    print(f"Loaded {len(variant_df)} variants")

    # Load metadata
    print("Loading track metadata...")
    try:
        metadata_df = pd.read_csv(METADATA_FILE)
        print(f"Loaded metadata for {len(metadata_df)} tracks")
    except Exception as e:
        print(f"Warning: Could not load metadata ({e}). BW signals will use track IDs.")
        metadata_df = None

    # Identify column types
    delta_cols = [c for c in variant_df.columns if c.startswith('D_BED_') or c.startswith('D_BW_')]
    bed_delta_cols = [c for c in delta_cols if c.startswith('D_BED_')]
    bw_delta_cols = [c for c in delta_cols if c.startswith('D_BW_')]

    print(f"Found {len(bed_delta_cols)} BED delta columns")
    print(f"Found {len(bw_delta_cols)} BW delta columns")



    # Ensure numeric
    print("Coercing delta columns to numeric...")
    for col in delta_cols:
        variant_df[col] = pd.to_numeric(variant_df[col], errors='coerce')

    # --- FAST VECTORIZED EXTRACTION ---
    print("Extracting signals (vectorized)...")
    signal_df = extract_signals_comprehensive_fast(
        variant_df, delta_cols, metadata_df, k=k_signals, variant_type=variant_type_arg
    )

    # Join results
    for col in signal_df.columns:
        variant_df[col] = signal_df[col].values

    # --- SAVE ---
    variant_df.to_parquet(OUTPUT_FILE)
    print(f"\nResults saved to: {OUTPUT_FILE}")

    return variant_df


if __name__ == "__main__":
    df = main("indel", PATHS, 10)


Loading data...
Loaded 11426 variants
Loading track metadata...
Loaded metadata for 7362 tracks
Found 21 BED delta columns
Found 3602 BW delta columns
Detected variant type: snp
Extracting comprehensive signals...


C:\Users\stavz\AppData\Local\Temp\ipykernel_32724\1372428149.py:408: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  variant_df[col] = signal_results[col]
C:\Users\stavz\AppData\Local\Temp\ipykernel_32724\1372428149.py:408: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  variant_df[col] = signal_results[col]
C:\Users\stavz\AppData\Local\Temp\ipykernel_32724\1372428149.py:408: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joi


Full results saved to: parquet/annotated_snps_signaled.parquet


# Defining variant and system prompts, setting up functions

In [ ]:
########################################################################
# ──────────────────────────────────────────────────────────────────────────────
# SYSTEM PROMPTS + VARIANT PROMPTS + FUNCTIONS
# ──────────────────────────────────────────────────────────────────────────────
########################################################################

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from typing import Optional
import gzip





# =============================================================================
# SYSTEM PROMPTS
# =============================================================================

SYSTEM_PROMPT_SNP = """You are an expert molecular geneticist and variant classifier specializing in the functional interpretation of genomic foundation model predictions. Your task is to evaluate whether computational signals from Nucleotide Transformer (NT) output for a variant aligns with the human-curated ClinVar rationale.

## INPUT DATA PER VARIANT

1. **Variant ID**: ClinVar variation identifier
2. **Region (Coarse Class)**: High-level genomic context derived from overlap annotations.
   - One of: CODING, SPLICE, UTR_5, UTR_3, PROMOTER, INTRONIC, GENIC_OTHER
   - This is a *contextual hint only*, not definitive evidence of mechanism.
   - BED and MLM signals take precedence over region when inferring mechanism.
3. **ClinVar Rationale** (FullRationale): Human-curated explanation of variant pathogenicity
4. **BED Feature Signals**: Delta predictions (alt - ref) for genomic annotations
   - Features include: splice_donor, splice_acceptor, exon, intron, ORF, start_codon, stop_codon, 5UTR, 3UTR, promoter, enhancer, CTCF, polyA_signal, etc.
   - Format: `feature=delta(ref=X)` where delta is the change and ref is the reference prediction
5. **Epigenomic and transcriptional Track Signals (BW)**: Delta predictions for tissue/cell-specific functional annotations
   - Format: `tissue|biosample|assay=delta(ref=X)`
   - These capture chromatin accessibility, histone marks, TF binding, RNA expression across tissues
6. **MLM Sequence Context**: Masked language model scores capturing sequence plausibility
   - LLR: Log-likelihood ratio log(P(ALT)/P(REF))
   - MLM_Delta: Change in model's sequence prediction confidence
   - MLM_Prior: Background sequence context score
   - LogProb_Delta: Change in sequence log probability **(Z-SCORED)**
   - LogProb_Ref: Log probability of reference sequence **(Z-SCORED)**

## CRITICAL: SIGNAL SCALES AND UNITS

The signal types operate on DIFFERENT scales. You must account for this when interpreting magnitudes:

### BED Features — Probability Scale [0, 1]
BED deltas represent changes in predicted **probabilities** of genomic annotations.
- Values are bounded: deltas range from -1.0 to +1.0
- A delta of -0.8 on a feature with ref=0.9 means the probability dropped from 90% to 10%
- **Negative delta (Loss)**: Variant disrupts the model's recognition of that genomic element
  - Example: `splice_donor=-0.8(ref=0.9)` → splice donor probability dropped from 90% to 10%
- **Positive delta (Gain)**: Variant creates or strengthens association with element
  - Example: `splice_donor=+0.6(ref=0.1)` → cryptic splice site gained (10% → 70%)

### Epigenomic and trasncriptional Tracks (BW) — Probability Scale [0, 1]
BW deltas represent changes in predicted **probabilities** of epigenomic activity (chromatin accessibility, histone marks, TF binding, RNA Expression).
- Values are bounded: deltas range from -1.0 to +1.0, same scale as BED features
- A delta of -0.6 on a feature with ref=0.8 means the predicted activity dropped from 80% to 20%
- BED and BW deltas are directly comparable in magnitude since both are on the probability scale
- **Context matters**: A BW signal is most informative when the tissue/cell type is relevant to the disease in question
  - Example: A strong delta in a blood/immune cell H3K27ac track is highly relevant for a blood disorder variant
  - Example: A strong delta in a brain-specific ATAC track is less relevant for a cardiac variant unless pleiotropic effects are expected

### MLM Scores — Log-Likelihood Scale (unbounded) + Z-Scored Metrics
- **LLR < -1**: Sequence context strongly disfavors the alt allele
- **LLR < -2**: Severe sequence disruption — the variant creates highly unusual sequence
- **MLM_Delta**: Large magnitude indicates significant sequence perturbation
- **LogProb_Delta (Z-SCORED)**: Z-score of log probability change.
- **LogProb_Ref (Z-SCORED)**: Z-score of reference sequence log probability. Indicates how typical the reference context is

**Important**: While NT doesn't explicitly model protein structure, the MLM component captures evolutionary sequence constraints that INCLUDE codon usage, amino acid conservation patterns, and protein-coding sequence signatures. A missense variant at a highly conserved residue will often show:
- Strong negative LLR (the alt codon/sequence is evolutionarily disfavored)
- Disruption in ORF or exon signals (even if subtle)

Therefore, MLM signals CAN provide indirect evidence for protein-level pathogenicity through sequence conservation patterns. Do not dismiss missense variants as "NOT_APPLICABLE" if MLM signals are strong.

## YOUR OUTPUT

Return a single JSON object with these fields:

```json
{
  "concordance": "CONCORDANT|PARTIAL|DISCORDANT|NOT_APPLICABLE",
  "explanation": "Brief reasoning (max 30 words) connecting signals to rationale",
  "signal_category": "STRONG|MODERATE|WEAK|ABSENT",
  "primary_signal_mechanism": "Inferred mechanism from signals",
  "key_signals": "Top 2-3 from each BED/BW/MLM features driving the interpretation",
  "rationale_mechanism": "What mechanism does ClinVar describe?",
  "nt_missed": true|false,
  "notes": "Optional additional context"
}
```

## OUTPUT RULES
- DO NOT mix primary_signal_mechanism which is from input signals and rationale_mechanism derived from Clinvar rationale.


## CONCORDANCE DEFINITIONS

- **CONCORDANT**: Signals support the mechanism described in the rationale
  - Rationale says "disrupts splicing" AND BED shows splice_donor/acceptor loss (large probability drop)
  - Rationale says "regulatory variant" AND BW shows tissue-relevant chromatin changes (large probability drops or gains)
  - Rationale describes conserved residue AND MLM shows strong negative LLR
  - Rationale is missing/minimal AND signals are absent (both uninformative = concordant by default)

- **PARTIAL**: Signals capture part but not all of the described mechanism
  - Rationale describes splice + protein effect, NT captures only splice
  - Some relevant signals present but weaker than expected
  - MLM suggests sequence disruption but BED signals are weak, when rationale describes a transcript level disruption.

- **DISCORDANT**: Signals contradict or fail to support the mechanism when they SHOULD
  - Rationale explicitly describes splice disruption but NO splice signals detected (BED probability unchanged)
  - Rationale describes highly conserved position but MLM shows no sequence constraint (LLR ~ 0)
  - Strong signals present but don't match the described mechanism at all

- **NOT_APPLICABLE**: Use sparingly — only when:
  - Rationale describes purely structural protein effects (e.g., "disrupts beta-sheet folding") with NO sequence conservation argument
  - AND MLM signals are neutral (LLR ~ 0, no sequence constraint detected)
  - If MLM shows strong negative LLR for a missense variant, it IS capturing evolutionary/functional constraint — use CONCORDANT or PARTIAL instead

## RULES
- Use expert biological reasoning
- Account for the different scales: BED and BW are both probability [0,1], MLM is log-likelihood (unbounded), and some metrics are Z-scored
- BED and BW deltas are on the same probability scale and can be compared directly
- Do NOT infer a mechanism solely from the Region field.
  - Region provides contextual plausibility; BED/BW/MLM signals determine mechanism.
- Consider tissue context when interpreting BW signals — disease-relevant tissues carry more weight
- MLM signals provide indirect protein-level evidence through sequence conservation — do not ignore them for missense variants
- A variant can have BOTH transcript and protein effects — capture what NT detects
- Prefer CONCORDANT/PARTIAL/DISCORDANT over NOT_APPLICABLE when ANY signal is informative
- Output only the JSON object, no additional text


"""


SYSTEM_PROMPT_INDEL = """You are an expert molecular geneticist and variant classifier specializing in the functional interpretation of genomic foundation model predictions. Your task is to evaluate whether computational signals from Nucleotide Transformer (NT) output for an INDEL variant's aligns with the human-curated ClinVar rationale.

## INPUT DATA PER VARIANT

1. **Variant ID**: ClinVar variation identifier
2. **Region**: Genomic region annotation
3. **ClinVar Rationale** (FullRationale): Human-curated explanation of variant pathogenicity
4. **BED Feature Signals**: Delta predictions (alt - ref) for genomic annotations
   - Features: splice_donor, splice_acceptor, exon, intron, ORF, start_codon, stop_codon, UTRs, regulatory elements
   - Format: `feature=delta(ref=X)`
5. **Epigenomic and transcriptional Track Signals (BW)**: Delta predictions for tissue/cell-specific functional annotations
   - Format: `tissue|biosample|assay=delta(ref=X)`
   - These capture chromatin accessibility, histone marks, TF binding, RNA expression across tissues
6. **MLM Sequence Context** (expanded for indels):
   - LLR: Log-likelihood ratio (alt vs ref sequence plausibility)
   - MLM_Delta: Sequence prediction confidence change
   - LogProb_Delta: Change in sequence log probability **(Z-SCORED)**
   - LogProb_Ref: Log probability of reference sequence **(Z-SCORED)**
   - EMB_cosine_dist: Embedding space distance (how much the sequence representation shifts)
   - EMB_l2_dist: L2 distance in embedding space **(Z-SCORED)**
   - EMB_max_pos_dist: Maximum position-wise embedding distance **(Z-SCORED)**
   - EMB_mean_pos_dist: Mean position-wise embedding distance **(Z-SCORED)**
   - MLM_KL_mean: Mean KL divergence of token distributions (average local sequence disruption)
   - MLM_KL_max: Max KL divergence of token distributions (peak local sequence disruption)

## CRITICAL: SIGNAL SCALES AND UNITS

The signal types operate on DIFFERENT scales. You must account for this when interpreting magnitudes:

### BED Features — Probability Scale [0, 1]
BED deltas represent changes in predicted **probabilities** of genomic annotations.
- Values are bounded: deltas range from -1.0 to +1.0
- A delta of -0.8 on a feature with ref=0.9 means the probability dropped from 90% to 10%


### Epigenomic Tracks and trasncriptional Tracks(BW) — Probability Scale [0, 1]
BW deltas represent changes in predicted **probabilities** of epigenomic activity (chromatin accessibility, histone marks, TF binding, RNA Expression).
- Values are bounded: deltas range from -1.0 to +1.0, same scale as BED features
- A delta of -0.6 on a feature with ref=0.8 means the predicted activity dropped from 80% to 20%
- BED and BW deltas are directly comparable in magnitude since both are on the probability scale
- **Context matters**: A BW signal is most informative when the tissue/cell type is relevant to the disease in question

### MLM / Embedding Scores — Various Scales (some Z-scored)
- **LLR**: Log-likelihood scale.
- **EMB_cosine_dist**: [0, 2] range (NOT z-scored)
- **MLM_KL_max**: Unbounded (NOT z-scored).
- **MLM_KL_mean**: Unbounded (NOT z-scored). Average disruption across positions
- **LogProb_Delta (Z-SCORED)**: Z-score of log probability change.
- **LogProb_Ref (Z-SCORED)**: Z-score of reference sequence log probability
- **EMB_l2_dist (Z-SCORED)**: Z-score of L2 embedding distance.
- **EMB_max_pos_dist (Z-SCORED)**: Z-score of maximum position-wise distance.
- **EMB_mean_pos_dist (Z-SCORED)**: Z-score of mean position-wise distance.

## INDEL-SPECIFIC INTERPRETATION

### Frameshift Detection
- Indels in coding regions often cause frameshifts leading to premature stop codons or nonsense-mediated decay
- Look for: strong ORF loss (BED probability drop), exon disruption, downstream stop codon effects
- BED signals may show: `ORF=-0.8, exon=-0.5` indicating loss of coding identity

### Splice Site Indels
- Insertions/deletions near splice junctions have strong effects
- Even small indels (1-2bp) can destroy canonical splice sites
- Look for splice_donor/acceptor losses with high magnitude (probability drops)

### Embedding Distance (EMB) — Critical for Indels
The embedding metrics capture how much the variant changes the overall sequence representation:
- **EMB_l2_dist, EMB_max_pos_dist, EMB_mean_pos_dist (Z-SCORED)**:

High EMB distance indicates the indel creates a sequence context the model recognizes as substantially different — this captures both structural (frameshift) and functional (protein-coding disruption) effects indirectly.

### MLM KL Divergence
- **MLM_KL_mean**: Average disruption; useful for assessing overall sequence perturbation
- Indicates the indel creates sequence context that violates learned patterns

### Protein-Level Effects via MLM/EMB
For in-frame indels affecting protein function, the MLM and embedding signals CAN capture disruption:
- High EMB_cosine_dist suggests the sequence change is functionally significant
- Strong negative LogProb_Delta (large negative z-score) indicates the alt sequence is evolutionarily disfavored
- These serve as proxies for protein constraint even without explicit protein modeling

## YOUR OUTPUT

Return a single JSON object:

```json
{
  "concordance": "CONCORDANT|PARTIAL|DISCORDANT|NOT_APPLICABLE",
  "explanation": "Brief reasoning (max 30 words)",
  "signal_category": "STRONG|MODERATE|WEAK|ABSENT",
  "primary_signal_mechanism": "Inferred mechanism from NT signals (e.g., 'frameshift_ORF_loss', 'splice_disruption', 'sequence_constraint_violation')",
  "key_signals": "Top 2-3 signals driving interpretation",
  "rationale_mechanism": "What mechanism does ClinVar describe?",
  "nt_missed": true|false,
  "embedding_impact": "HIGH|MODERATE|LOW|NONE",
  "notes": "Optional context"
}
```

## OUTPUT RULES
- DO NOT mix primary_signal_mechanism which is from input signals and rationale_mechanism derived from Clinvar rationale.


## CONCORDANCE DEFINITIONS

- **CONCORDANT**: Signals support the mechanism
  - Frameshift rationale + strong ORF/exon probability loss + high EMB distance
  - Splice site indel + splice signal probability loss
  - In-frame deletion at conserved region + high EMB_cosine_dist + strong negative LogProb_Delta z-score
  - Rationale missing AND signals absent (both uninformative = concordant by default)

- **PARTIAL**: Partial signal support
  - Some signals align but magnitude lower than expected for their respective scales
  - EMB shows disruption but BED probabilities are unchanged (or vice versa)
  - Only one signal type supports the rationale

- **DISCORDANT**: Signals contradict rationale
  - Rationale describes frameshift/splice disruption but BED probabilities + EMB signals are weak
  - Strong signals present but mechanism doesn't match at all
  - Rationale describes severe effect but EMB_cosine_dist is low and BED probabilities unchanged

- **NOT_APPLICABLE**: Use sparingly — only when:
  - Rationale describes purely structural protein effects with no sequence conservation argument
  - AND all signals are neutral (low EMB, neutral MLM, no BED probability changes)
  - If EMB or MLM shows significant change, use CONCORDANT/PARTIAL/DISCORDANT instead

## RULES
- Indels often have larger effects than SNPs — calibrate expectations accordingly
- Account for the different scales: BED and BW are both probability [0,1], MLM/EMB have their own scales
- BED and BW deltas are on the same probability scale and can be compared directly
- EMB and MLM signals provide indirect protein-level evidence — high values suggest functional impact
- Consider tissue context when interpreting BW signals — disease-relevant tissues carry more weight
- Consider BOTH BED (structural annotation probabilities) AND EMB/MLM (sequence context) signals
- Prefer CONCORDANT/PARTIAL/DISCORDANT over NOT_APPLICABLE when ANY signal shows meaningful change
- Output only the JSON object
"""



# =============================================================================
# VARIANT PROMPT BUILDERS
# =============================================================================

def make_variant_prompt_snp(row: pd.Series) -> str:
    """
    Build a per-variant prompt for SNP concordance judgment.
    """
    def safe_get(col, default="(not available)"):
        v = row.get(col, default)
        if pd.isna(v) or v == "":
            return default
        return str(v)

    # Core identifiers
    variant_id = row.name if row.name else safe_get('#VariationID', 'Unknown')
    region = safe_get('region', 'Unknown')
    region_class = safe_get('region_class', 'Unknown')

    # ClinVar rationale
    rationale = safe_get('FullRationale')
    if rationale == "(not available)" or rationale.strip() == "":
        rationale = "(No detailed rationale provided)"
    elif len(rationale) > 1500:
        rationale = rationale[:1500] + "... [truncated]"

    # BED signals
    bed_abs = safe_get('BED_Top_Abs', 'None')
    bed_gains = safe_get('BED_Top_Gains', 'None')
    bed_losses = safe_get('BED_Top_Losses', 'None')

    # BW signals
    bw_abs = safe_get('BW_Top_Abs', 'None')
    bw_gains = safe_get('BW_Top_Gains', 'None')
    bw_losses = safe_get('BW_Top_Losses', 'None')

    # MLM signals
    mlm_summary = safe_get('MLM_Summary', 'None')


    # ****** individual variant PROMPT ******
    prompt = f"""## Variant: {variant_id}
**Region:** {region_class}

### ClinVar Rationale
{rationale}

### BED Feature Signals (Genomic Annotations)
**Top by magnitude:** {bed_abs}
**Top gains (positive delta):** {bed_gains}
**Top losses (negative delta):** {bed_losses}

### Epigenomic Track Signals (BW)
**Top by magnitude:** {bw_abs}
**Top gains:** {bw_gains}
**Top losses:** {bw_losses}

### MLM Sequence Context
{mlm_summary}

---
Evaluate concordance between the NT signals and the ClinVar rationale. Return JSON only."""

    return prompt


def make_variant_prompt_indel(row: pd.Series) -> str:
    """
    Build a per-variant prompt for INDEL concordance judgment.
    Includes expanded MLM metrics relevant to indels.
    """
    def safe_get(col, default="(not available)"):
        v = row.get(col, default)
        if pd.isna(v) or v == "":
            return default
        if isinstance(v, float):
            return f"{v:.4f}"
        return str(v)

    # Core identifiers
    variant_id = row.name if row.name else safe_get('#VariationID', 'Unknown')
    region = safe_get('region', 'Unknown')
    indel_size = row.get('indel_size', 'Unknown')
    variant_type = row.get('variant_type', 'Unknown')

    # ClinVar rationale
    rationale = safe_get('FullRationale')
    if rationale == "(not available)" or rationale.strip() == "":
        rationale = "(No detailed rationale provided)"
    elif len(rationale) > 1500:
        rationale = rationale[:1500] + "... [truncated]"

    # BED signals
    bed_abs = safe_get('BED_Top_Abs', 'None')
    bed_gains = safe_get('BED_Top_Gains', 'None')
    bed_losses = safe_get('BED_Top_Losses', 'None')

    # BW signals
    bw_abs = safe_get('BW_Top_Abs', 'None')
    bw_gains = safe_get('BW_Top_Gains', 'None')
    bw_losses = safe_get('BW_Top_Losses', 'None')

    # MLM summary
    mlm_summary = safe_get('MLM_Summary', 'None')

    # Expanded MLM metrics for indels
    mlm_logprob_delta = safe_get('MLM_logprob_delta', 'N/A')
    emb_cosine = safe_get('EMB_cosine_dist', 'N/A')
    emb_l2 = safe_get('EMB_l2_dist', 'N/A')
    kl_max = safe_get('MLM_KL_max', 'N/A')
    kl_mean = safe_get('MLM_KL_mean', 'N/A')

  # ****** individual variant PROMPT ******
    prompt = f"""## Variant: {variant_id}
**Variant Type:** {variant_type}
**Variant Size:** {indel_size}
**Region:** {region}

### ClinVar Rationale
{rationale}

### BED Feature Signals (Genomic Annotations)
**Top by magnitude:** {bed_abs}
**Top gains (positive delta):** {bed_gains}
**Top losses (negative delta):** {bed_losses}

### Epigenomic Track Signals (BW)
**Top by magnitude:** {bw_abs}
**Top gains:** {bw_gains}
**Top losses:** {bw_losses}

### MLM Sequence Context
**Summary:** {mlm_summary}

**Detailed Indel Metrics:**
- Log-probability delta: {mlm_logprob_delta}
- Embedding cosine distance: {emb_cosine}
- Embedding L2 distance: {emb_l2}
- KL divergence (max): {kl_max}
- KL divergence (mean): {kl_mean}

---
Evaluate concordance between the NT signals and the ClinVar rationale. Return JSON only."""

    return prompt


def make_variant_prompt(row: pd.Series, variant_type: str = 'snp') -> str:
    """
    Dispatch to appropriate prompt builder based on variant type.
    """
    if variant_type == 'indel':
        return make_variant_prompt_indel(row)
    else:
        return make_variant_prompt_snp(row)


# =============================================================================
# PROMPT TABLE BUILDER
# =============================================================================

def build_prompts_table(df: pd.DataFrame, variant_type: str = 'snp') -> pd.DataFrame:
    """
    Add 'llm_prompt' column to dataframe.

    Args:
        df: DataFrame with signal columns
        variant_type: 'snp' or 'indel'

    Returns:
        DataFrame with added 'llm_prompt' column
    """
    out = df.copy()
    out['llm_prompt'] = out.apply(
        lambda row: make_variant_prompt(row, variant_type),
        axis=1
    )
    return out


def get_system_prompt(variant_type: str = 'snp') -> str:
    """
    Return appropriate system prompt for variant type.
    """
    if variant_type == 'indel':
        return SYSTEM_PROMPT_INDEL
    else:
        return SYSTEM_PROMPT_SNP



# =============================================================================
# EXAMPLE USAGE
# =============================================================================

# if __name__ == "__main__":
    # Example: Load and process SNPs

    # Load SNPs
    # print("Loading SNP data...")
    # snp_df = pq.read_table(PATHS["snp_annotated_signaled"]).to_pandas()
    # print(f"Loaded {len(snp_df)} SNPs")

    # # Build prompts
    # snp_with_prompts = build_prompts_table(snp_df, variant_type='snp')


    # # Load indels
    # print("Loading INDEL data...")
    # indel_df = pq.read_table(PATHS["indel_annotated_signaled"]).to_pandas()
    # print(f"Loaded {len(indel_df)} INDELs")

    # # Build prompts
    # indel_with_prompts = build_prompts_table(indel_df, variant_type='indel')


# **ONE BY ONE API ACCESS**

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import re
from typing import Optional, Dict, Any, List
from google import genai
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from google.api_core import exceptions



# =============================================================================
# API CALL WITH RETRY
# =============================================================================

_thread_local = threading.local()

def get_thread_client(api_key: str):
    # One client per thread
    if not hasattr(_thread_local, "client"):
        from google import genai
        _thread_local.client = genai.Client(api_key=api_key)
    return _thread_local.client

def call_llm_api_robust_threaded(
    user_content: str,
    system_content: str,
    model_name: str,
    api_key: str,
    max_retries: int = 10,
):

    client = get_thread_client(api_key)

    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model=model_name,
                contents=user_content,
                config={"system_instruction": system_content},
            )
            return resp.text.strip() if resp and resp.text else ""
        except exceptions.ResourceExhausted:
            wait_time = (2 ** attempt) * 5
            print(f"\n[Rate Limit] sleep {wait_time}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait_time)
        except exceptions.InvalidArgument as e:
            print(f"\n[Invalid Request] {e}")
            return None
        except Exception as e:
            print(f"\n[Error] {e}")
            time.sleep(5)

    return None


# =============================================================================
# JSON PARSING
# =============================================================================

def parse_llm_json_response(response_text: str) -> Optional[Dict[str, Any]]:
    """
    Parse JSON from LLM response, handling common formatting issues.

    LLMs sometimes wrap JSON in markdown code blocks or add extra text.
    This function extracts and parses the JSON robustly.
    """
    if not response_text:
        return None

    # Try direct parse first
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass

    # Try extracting from markdown code block
    # Matches ```json ... ``` or ``` ... ```
    code_block_pattern = r'```(?:json)?\s*\n?(.*?)\n?```'
    matches = re.findall(code_block_pattern, response_text, re.DOTALL)

    for match in matches:
        try:
            return json.loads(match.strip())
        except json.JSONDecodeError:
            continue

    # Try finding JSON object pattern { ... }
    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = re.findall(json_pattern, response_text, re.DOTALL)

    for match in matches:
        try:
            return json.loads(match)
        except json.JSONDecodeError:
            continue

    # Failed to parse
    return None


def validate_evaluation_response(parsed: Dict[str, Any], variant_type: str = 'snp') -> Dict[str, Any]:
    """
    Validate and normalize the parsed JSON response.
    Fills in defaults for missing fields.
    """
    # Expected fields
    expected_fields_base = {
        'concordance': 'PARSE_ERROR',
        'explanation': '',
        'signal_category': 'UNKNOWN',
        'primary_mechanism': '',
        'key_signals': '',
        'rationale_mechanism': '',
        'nt_missed': None,
        'notes': ''
    }

    # Add indel-specific field
    if variant_type == 'indel':
        expected_fields_base['embedding_impact'] = 'UNKNOWN'

    # Valid concordance values
    valid_concordance = {'CONCORDANT', 'PARTIAL', 'DISCORDANT', 'NOT_APPLICABLE'}
    valid_signal_category = {'STRONG', 'MODERATE', 'WEAK', 'ABSENT', 'UNKNOWN'}
    valid_embedding_impact = {'HIGH', 'MODERATE', 'LOW', 'NONE', 'UNKNOWN'}

    # Build result with defaults
    result = {}
    for field, default in expected_fields_base.items():
        result[field] = parsed.get(field, default)

    # Normalize concordance
    if result['concordance'] not in valid_concordance:
        result['concordance'] = 'PARSE_ERROR'

    # Normalize signal_category
    if result['signal_category'] not in valid_signal_category:
        result['signal_category'] = 'UNKNOWN'

    # Normalize embedding_impact for indels
    if variant_type == 'indel' and result.get('embedding_impact') not in valid_embedding_impact:
        result['embedding_impact'] = 'UNKNOWN'

    # Ensure nt_missed is boolean or None
    if result['nt_missed'] is not None:
        result['nt_missed'] = bool(result['nt_missed'])

    return result


# =============================================================================
# SINGLE VARIANT EVALUATION
# =============================================================================

def evaluate_single_variant_parallel(row, system_prompt, variant_type, model_name, api_key):
    # Same as your evaluate_single_variant, but swap the call func:
    variation_id = row.get('#VariationID', row.name)
    user_prompt = row.get('llm_prompt', '')
    if not user_prompt:
        return {
            '#VariationID': variation_id,
            'concordance': 'ERROR',
            'explanation': 'No prompt available',
            'signal_category': 'UNKNOWN',
            'primary_mechanism': '',
            'key_signals': '',
            'rationale_mechanism': '',
            'nt_missed': None,
            'notes': 'Missing llm_prompt column',
            'raw_response': None,
            'parse_success': False
        }

    raw_response = call_llm_api_robust_threaded(
        user_content=user_prompt,
        system_content=system_prompt,
        model_name=model_name,
        api_key=api_key,
        max_retries=10
    )

    # Reuse your parsing/validation functions:
    if raw_response is None:
        return {
            '#VariationID': variation_id,
            'concordance': 'API_ERROR',
            'explanation': 'API call failed',
            'signal_category': 'UNKNOWN',
            'primary_mechanism': '',
            'key_signals': '',
            'rationale_mechanism': '',
            'nt_missed': None,
            'notes': 'API returned None',
            'raw_response': None,
            'parse_success': False
        }

    parsed = parse_llm_json_response(raw_response)
    if parsed is None:
        return {
            '#VariationID': variation_id,
            'concordance': 'PARSE_ERROR',
            'explanation': 'Failed to parse JSON',
            'signal_category': 'UNKNOWN',
            'primary_mechanism': '',
            'key_signals': '',
            'rationale_mechanism': '',
            'nt_missed': None,
            'notes': f'Raw response: {raw_response[:500]}',
            'raw_response': raw_response,
            'parse_success': False
        }

    validated = validate_evaluation_response(parsed, variant_type)
    validated['#VariationID'] = variation_id
    validated['raw_response'] = raw_response
    validated['parse_success'] = True
    return validated




# =============================================================================
# BATCH EVALUATION
# =============================================================================


def evaluate_variants_parallel(
    df: pd.DataFrame,
    system_prompt: str,
    variant_type: str,
    model_name: str,
    api_key: str,
    max_workers: int = 10,              # <-- key knob
    save_checkpoint_every: int = 50,
    checkpoint_path: str = None,
    resume_from_checkpoint: bool = True,
):
    results = []
    evaluated_ids = set()

    # Resume
    if checkpoint_path and resume_from_checkpoint:
        try:
            ck = pd.read_parquet(checkpoint_path)
            results = ck.to_dict("records")
            evaluated_ids = set(ck["#VariationID"].astype(str))
            print(f"Resumed {len(evaluated_ids)} already done")
        except FileNotFoundError:
            pass

    # Build jobs for remaining rows
    remaining = []
    for row_idx, row in df.iterrows():
        vid = str(row.get("#VariationID", row_idx))
        if vid not in evaluated_ids:
            remaining.append((row_idx, row))

    total_target = len(evaluated_ids) + len(remaining)
    done_counter = 0
    start = time.time()

    # Submit + collect
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        future_to_vid = {}
        for row_idx, row in remaining:
            vid = str(row.get("#VariationID", row_idx))
            fut = ex.submit(
                evaluate_single_variant_parallel,
                row, system_prompt, variant_type, model_name, api_key
            )
            future_to_vid[fut] = vid

        for fut in as_completed(future_to_vid):
            vid = future_to_vid[fut]
            try:
                res = fut.result()
            except Exception as e:
                res = {
                    "#VariationID": vid,
                    "concordance": "ERROR",
                    "explanation": f"Worker exception: {e}",
                    "signal_category": "UNKNOWN",
                    "primary_mechanism": "",
                    "key_signals": "",
                    "rationale_mechanism": "",
                    "nt_missed": None,
                    "notes": "Worker crashed",
                    "raw_response": None,
                    "parse_success": False,
                }

            results.append(res)
            done_counter += 1

            elapsed = time.time() - start
            rate_per_min = (done_counter / elapsed) * 60 if elapsed > 0 else 0
            status_char = "✓" if res.get("parse_success") else "✗"
            print(f"\r[{len(results)}/{total_target}] {status_char} {vid} | {rate_per_min:.1f}/min",
                  end="", flush=True)

            # checkpoint safely from the main thread only
            if checkpoint_path and (len(results) % save_checkpoint_every == 0):
                pd.DataFrame(results).to_parquet(checkpoint_path)
                print(f"\n[Checkpoint] saved {len(results)} to {checkpoint_path}")

    print(f"\nDone. Total results: {len(results)}")
    out = pd.DataFrame(results)
    if checkpoint_path:
        out.to_parquet(checkpoint_path)
    return out


# =============================================================================
# RESULTS ANALYSIS
# =============================================================================

def summarize_evaluation_results(results_df: pd.DataFrame) -> Dict[str, Any]:
    """
    Generate summary statistics from evaluation results.
    """
    total = len(results_df)

    summary = {
        'total_variants': total,
        'parse_success_rate': results_df['parse_success'].mean() if 'parse_success' in results_df.columns else None,
        'concordance_distribution': results_df['concordance'].value_counts().to_dict(),
        'signal_category_distribution': results_df['signal_category'].value_counts().to_dict(),
        'nt_missed_rate': results_df['nt_missed'].mean() if 'nt_missed' in results_df.columns else None,
    }

    # Concordance percentages
    concordance_counts = results_df['concordance'].value_counts()
    for cat in ['CONCORDANT', 'PARTIAL', 'DISCORDANT', 'NOT_APPLICABLE']:
        count = concordance_counts.get(cat, 0)
        summary[f'pct_{cat.lower()}'] = count / total * 100 if total > 0 else 0

    return summary


def print_evaluation_summary(results_df: pd.DataFrame) -> None:
    """
    Print a formatted summary of evaluation results.
    """
    summary = summarize_evaluation_results(results_df)

    print("=" * 60)
    print("EVALUATION SUMMARY")
    print("=" * 60)
    print(f"Total variants evaluated: {summary['total_variants']}")
    print(f"Parse success rate: {summary['parse_success_rate']*100:.1f}%")
    print()
    print("Concordance Distribution:")
    print("-" * 30)
    for cat in ['CONCORDANT', 'PARTIAL', 'DISCORDANT', 'NOT_APPLICABLE']:
        pct = summary.get(f'pct_{cat.lower()}', 0)
        count = summary['concordance_distribution'].get(cat, 0)
        bar = '█' * int(pct / 2)
        print(f"  {cat:16} {count:4} ({pct:5.1f}%) {bar}")

    # Errors
    error_cats = ['API_ERROR', 'PARSE_ERROR', 'ERROR']
    error_count = sum(summary['concordance_distribution'].get(c, 0) for c in error_cats)
    if error_count > 0:
        print(f"\n  Errors: {error_count}")

    print()
    print("Signal Category Distribution:")
    print("-" * 30)
    for cat, count in summary['signal_category_distribution'].items():
        pct = count / summary['total_variants'] * 100
        print(f"  {cat:12} {count:4} ({pct:5.1f}%)")

    if summary['nt_missed_rate'] is not None:
        print(f"\nNT Missed Rate: {summary['nt_missed_rate']*100:.1f}%")

    print("=" * 60)


# =============================================================================
# MAIN EVALUATION PIPELINE
# =============================================================================

def run_evaluation_pipeline(
    df_with_prompts: pd.DataFrame,
    variant_type: str = 'snp',
    output_path: str = None,
    checkpoint_path: str = None,
    model_name: str = MODEL_NAME,
    api_key: str = API_KEY,
    delay: float = 0.5,
    sample_size: int = None
) -> pd.DataFrame:
    """
    Run the full evaluation pipeline.

    Args:
        df_with_prompts: DataFrame with 'llm_prompt' column
        variant_type: 'snp' or 'indel'
        output_path: Path to save final results
        checkpoint_path: Path for checkpointing
        model_name: LLM model
        api_key: API key
        delay: Delay between API calls
        sample_size: If set, evaluate only this many variants

    Returns:
        DataFrame with evaluation results
    """


    # Get system prompt
    system_prompt = get_system_prompt(variant_type)

    # Sample if requested
    if sample_size and sample_size < len(df_with_prompts):
        df_eval = df_with_prompts.sample(n=sample_size, random_state=42)
        print(f"Sampled {sample_size} variants for evaluation")
    else:
        df_eval = df_with_prompts

    print(f"Starting evaluation of {len(df_eval)} {variant_type} variants")
    print(f"Model: {model_name}")
    print(f"Delay between calls: {delay}s")
    print()

    # Run evaluation
    results_df = evaluate_variants_parallel(
        df=df_eval,
        system_prompt=system_prompt,
        variant_type=variant_type,
        model_name=model_name,
        api_key=api_key,
        checkpoint_path=checkpoint_path
    )

    # Print summary
    print_evaluation_summary(results_df)

    # Save final results
    if output_path:
        results_df.to_parquet(output_path)
        print(f"\nResults saved to: {output_path}")

        # Also save CSV for easy viewing
        csv_path = output_path.replace('.parquet', '.csv')
        results_df.to_csv(csv_path, index=False)
        print(f"CSV saved to: {csv_path}")

    return results_df


# =============================================================================
# MERGE RESULTS BACK TO ORIGINAL DATA
# =============================================================================

def merge_evaluation_results(
    original_df: pd.DataFrame,
    results_df: pd.DataFrame,
    on: str = '#VariationID'
) -> pd.DataFrame:
    """
    Merge evaluation results back to the original variant DataFrame.

    Args:
        original_df: Original DataFrame with variant data
        results_df: Evaluation results DataFrame
        on: Column to merge on

    Returns:
        Merged DataFrame
    """
    # Select columns to merge (exclude raw_response to save space)
    merge_cols = [on, 'concordance', 'explanation', 'signal_category',
                  'primary_mechanism', 'key_signals', 'rationale_mechanism',
                  'nt_missed', 'parse_success']

    # Add embedding_impact if present (indels)
    if 'embedding_impact' in results_df.columns:
        merge_cols.append('embedding_impact')

    results_slim = results_df[[c for c in merge_cols if c in results_df.columns]].copy()

    # Ensure merge column types match
    original_df[on] = original_df[on].astype(str)
    results_slim[on] = results_slim[on].astype(str)

    merged = original_df.merge(results_slim, on=on, how='left', suffixes=('', '_eval'))

    return merged


# =============================================================================
# EXAMPLE USAGE
# =============================================================================

if __name__ == "__main__":
    import pyarrow.parquet as pq

    # Configuration
    PATHS_LLM = {
        "snp_annotated_signaled": "parquet/annotated_snps_signaled.parquet",
        "indel_annotated_signaled": "parquet/annotated_indels_signaled.parquet",
        "snp_evaluation_results": "parquet/snp_evaluation_results.parquet_PARALLEL",
        "snp_evaluation_checkpoint": "parquet/snp_evaluation_checkpoint_PARALLEL_2.parquet",
        "indel_evaluation_checkpoint": "parquet/indel_evaluation_checkpoint_PARALLEL_erros.parquet"
    }

    API_KEY = "REDACTED_API_KEY"

    # Load data
    print("Loading data...")
    indel_df = pq.read_table(PATHS_LLM["indel_annotated_signaled"]).to_pandas()
    print(f"Loaded {len(indel_df)} variants")

    # Build prompts
    print("Building prompts...")
    snp_with_prompts = build_prompts_table(indel_df, variant_type='indel')

    output_path = f"parquet/indel_evaluation_results_trial.parquet"

    # Run evaluation
    results = run_evaluation_pipeline(
        df_with_prompts=snp_with_prompts,
    variant_type='indel',
    output_path=output_path,
    checkpoint_path=PATHS_LLM["indel_evaluation_checkpoint"],
    model_name="gemini-3-flash-preview",
    api_key=API_KEY,
    delay=0.5,
    # sample_size=50
)

    # Merge back to original - OPTIONAL
    # snp_with_eval = merge_evaluation_results(snp_df, results)
    # snp_with_eval.to_parquet("parquet/snps_with_evaluation.parquet")

    # print("\nDone!")

#  A BATCH API access for LLM evaluation

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import re
from typing import Optional, Dict, Any, List
from pathlib import Path
from google.genai import types
from google import genai as genai_client




# =============================================================================
# BATCH FILE PREPARATION
# =============================================================================

def prepare_batch_jsonl(
    df_with_prompts: pd.DataFrame,
    system_prompt: str,
    output_path: str,
    variant_type: str = 'snp'
) -> str:
    """
    Prepare JSONL file for batch API submission.

    Each line follows the format required by Gemini Batch API:
    {"request": {"contents": [...], "system_instruction": {...}}, "custom_id": "..."}

    Args:
        df_with_prompts: DataFrame with 'llm_prompt' and '#VariationID' columns
        system_prompt: System instruction text
        output_path: Path for output JSONL file
        variant_type: 'snp' or 'indel'

    Returns:
        Path to created JSONL file
    """
    batch_requests = []

    for idx, row in df_with_prompts.iterrows():
        variation_id = str(row.get('#VariationID', row.name))
        user_prompt = row.get('llm_prompt', '')

        if not user_prompt:
            continue

        # Build request in Gemini batch format
        request = {
            "custom_id": variation_id,
            "request": {
                "model": f'models/{BATCH_MODEL}',
                "contents": [
                    {
                        "role": "user",
                        "parts": [{"text": user_prompt}]
                    }
                ],
                "system_instruction": {
                    "parts": [{"text": system_prompt}]
                },
                "generation_config": {
                    "temperature": 0.1,  # Low temperature for consistent JSON
                    "max_output_tokens": 4096,
                    "response_mime_type": "application/json"  # Request JSON output
                }
            }
        }

        batch_requests.append(request)

    # Write JSONL
    with open(output_path, 'w') as f:
        for req in batch_requests:
            f.write(json.dumps(req) + '\n')

    print(f"Created batch file: {output_path}")
    print(f"Total requests: {len(batch_requests)}")

    return output_path


def prepare_batch_jsonl_simple(
    df_with_prompts: pd.DataFrame,
    system_prompt: str,
    output_path: str
) -> str:
    """
    Simpler JSONL format - just the essentials.
    Adjust based on actual Gemini Batch API requirements.
    """
    with open(output_path, 'w') as f:
        for idx, row in df_with_prompts.iterrows():
            variation_id = str(row.get('#VariationID', row.name))
            user_prompt = row.get('llm_prompt', '')

            if not user_prompt:
                continue

            entry = {
                "custom_id": variation_id,
                "body": {
                    "contents": [
                        {"role": "user", "parts": [{"text": user_prompt}]}
                    ],
                    "systemInstruction": {
                        "parts": [{"text": system_prompt}]
                    },
                    "generationConfig": {
                        "temperature": 0.1,
                        "maxOutputTokens": 1024
                    }
                }
            }

            f.write(json.dumps(entry) + '\n')

    print(f"Created batch file: {output_path}")
    return output_path


# =============================================================================
# BATCH JOB MANAGEMENT
# =============================================================================

def upload_and_submit_batch(
    jsonl_path: str,
    display_name: str,
    api_key: str = None,
    model: str = "gemini-3-flash-preview"
) -> Dict[str, Any]:
    """
    Upload JSONL file and submit batch job.

    Args:
        jsonl_path: Path to prepared JSONL file
        display_name: Name for the batch job
        api_key: API key
        model: Model to use

    Returns:
        Dict with batch job info
    """
    client = genai_client.Client(api_key=api_key)

    # Upload the file
    print(f"Uploading {jsonl_path}...")
    uploaded_file = client.files.upload(
    file=jsonl_path,
    config=types.UploadFileConfig(mime_type="application/jsonl")
    )
    print(f"Uploaded: {uploaded_file.name}")

    # Create batch job
    print(f"Creating batch job...")
    batch_job = client.batches.create(
        model=model,
        src=uploaded_file.name,
        config={'display_name': display_name}
    )

    print(f"Batch job created: {batch_job.name}")
    print(f"Status: {batch_job.state}")

    return {
        'job_name': batch_job.name,
        'file_name': uploaded_file.name,
        'display_name': display_name,
        'state': str(batch_job.state),
        'created_time': time.time()
    }


def check_batch_status(
    job_name: str,
    api_key: str = None
) -> Dict[str, Any]:
    """
    Check status of a batch job.

    Args:
        job_name: Name of the batch job
        api_key: API key

    Returns:
        Dict with status info
    """
    client = genai_client.Client(api_key=api_key)

    batch_job = client.batches.get(name=job_name)

    status = {
        'name': batch_job.name,
        'state': str(batch_job.state),
        'display_name': getattr(batch_job, 'display_name', ''),
    }

    # Add progress info if available
    if hasattr(batch_job, 'request_counts'):
        counts = batch_job.request_counts
        status['total_requests'] = getattr(counts, 'total', 0)
        status['succeeded'] = getattr(counts, 'succeeded', 0)
        status['failed'] = getattr(counts, 'failed', 0)
        status['pending'] = getattr(counts, 'pending', 0)

    return status


def wait_for_batch_completion(
    job_name: str,
    api_key: str = None,
    poll_interval: int = 60,
    timeout: int = 7200  # 2 hours
) -> Dict[str, Any]:
    """
    Wait for batch job to complete, polling periodically.

    Args:
        job_name: Name of the batch job
        api_key: API key
        poll_interval: Seconds between status checks
        timeout: Maximum seconds to wait

    Returns:
        Final status dict
    """
    start_time = time.time()

    print(f"Waiting for batch job: {job_name}")
    print(f"Polling every {poll_interval}s, timeout {timeout}s")

    while True:
        status = check_batch_status(job_name, api_key)
        # state = status['state']
        state = str(status['state'])
        print(state)

        elapsed = time.time() - start_time

        # Progress display
        if 'total_requests' in status:
            succeeded = status.get('succeeded', 0)
            total = status.get('total_requests', 0)
            pct = (succeeded / total * 100) if total > 0 else 0
            print(f"\r[{elapsed/60:.1f}min] State: {state} | "
                  f"Progress: {succeeded}/{total} ({pct:.1f}%)", end='', flush=True)
        else:
            print(f"\r[{elapsed/60:.1f}min] State: {state}", end='', flush=True)

        # Check completion states
        if state in ['JobState.JOB_STATE_SUCCEEDED', 'JOB_STATE_SUCCEEDED', 'SUCCEEDED', 'STATE_SUCCEEDED']:
            print(f"\n✓ Batch job completed successfully!")
            return status

        if state in ['JOB_STATE_FAILED', 'FAILED', 'STATE_FAILED']:
            print(f"\n✗ Batch job failed!")
            return status

        if state in ['JOB_STATE_CANCELLED', 'CANCELLED', 'STATE_CANCELLED']:
            print(f"\n✗ Batch job was cancelled!")
            return status

        # Check timeout
        if elapsed > timeout:
            print(f"\n⚠ Timeout reached ({timeout}s)")
            return status

        time.sleep(poll_interval)


def download_batch_results(job_name: str, output_path: str, api_key: str = None) -> str:
    client = genai_client.Client(api_key=api_key)

    # 1. Retrieve the job
    batch_job = client.batches.get(name=job_name)

    # 2. Extract the filename from the .dest object
    # Based on your diagnostic, the attribute is 'file_name'
    result_file_name = None

    if hasattr(batch_job, 'dest') and batch_job.dest:
        result_file_name = batch_job.dest.file_name

    # Safety check if it's still None
    if not result_file_name:
        raise ValueError(f"Could not find 'file_name' in batch_job.dest. Object dump: {batch_job}")

    print(f"Downloading results from: {result_file_name}")

    # 3. Download using the 'file' keyword argument
    # This returns the raw bytes
    result_bytes = client.files.download(file=result_file_name)

    # 4. Save to disk
    with open(output_path, 'wb') as f:
        f.write(result_bytes)

    print(f"✓ Results saved to: {output_path}")
    return output_path


# =============================================================================
# RESULTS PARSING
# =============================================================================

def parse_llm_json_response(response_text: str) -> Optional[Dict[str, Any]]:
    """
    Parse JSON from LLM response, handling common formatting issues.
    """
    if not response_text:
        return None

    # Try direct parse first
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass

    # Try extracting from markdown code block
    code_block_pattern = r'```(?:json)?\s*\n?(.*?)\n?```'
    matches = re.findall(code_block_pattern, response_text, re.DOTALL)

    for match in matches:
        try:
            return json.loads(match.strip())
        except json.JSONDecodeError:
            continue

    # Try finding JSON object pattern { ... }
    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = re.findall(json_pattern, response_text, re.DOTALL)

    for match in matches:
        try:
            return json.loads(match)
        except json.JSONDecodeError:
            continue

    return None


def parse_batch_results(
    results_path: str,
    variant_type: str = 'snp'
) -> pd.DataFrame:
    """
    Parse batch results JSONL file into DataFrame.

    Args:
        results_path: Path to results JSONL file
        variant_type: 'snp' or 'indel' (affects expected fields)

    Returns:
        DataFrame with parsed evaluation results
    """
    results = []

    with open(results_path, 'r') as f:
        for line_num, line in enumerate(f):
            if not line.strip():
                continue

            try:
                entry = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Warning: Failed to parse line {line_num}: {e}")
                continue

            # Extract custom_id (VariationID)
            custom_id = entry.get('custom_id', f'unknown_{line_num}')

            # Extract response
            response = entry.get('response', {})

            # Check for errors
            if 'error' in response:
                results.append({
                    '#VariationID': custom_id,
                    'concordance': 'API_ERROR',
                    'explanation': str(response.get('error', '')),
                    'signal_category': 'UNKNOWN',
                    'primary_mechanism': '',
                    'key_signals': '',
                    'rationale_mechanism': '',
                    'nt_missed': None,
                    'raw_response': json.dumps(response),
                    'parse_success': False
                })
                continue

            # Extract text from response
            # Structure varies - adapt based on actual API response format
            try:
                # Try common response structures
                if 'candidates' in response:
                    text = response['candidates'][0]['content']['parts'][0]['text']
                elif 'content' in response:
                    text = response['content']['parts'][0]['text']
                elif 'text' in response:
                    text = response['text']
                elif 'body' in entry:
                    # Alternative format
                    body = entry['body']
                    if 'candidates' in body:
                        text = body['candidates'][0]['content']['parts'][0]['text']
                    else:
                        text = str(body)
                else:
                    text = json.dumps(response)
            except (KeyError, IndexError, TypeError) as e:
                results.append({
                    '#VariationID': custom_id,
                    'concordance': 'PARSE_ERROR',
                    'explanation': f'Could not extract text: {e}',
                    'signal_category': 'UNKNOWN',
                    'primary_mechanism': '',
                    'key_signals': '',
                    'rationale_mechanism': '',
                    'nt_missed': None,
                    'raw_response': json.dumps(entry),
                    'parse_success': False
                })
                continue

            # Parse the JSON from the text
            parsed = parse_llm_json_response(text)

            if parsed is None:
                results.append({
                    '#VariationID': custom_id,
                    'concordance': 'PARSE_ERROR',
                    'explanation': 'Failed to parse JSON from response',
                    'signal_category': 'UNKNOWN',
                    'primary_mechanism': '',
                    'key_signals': '',
                    'rationale_mechanism': '',
                    'nt_missed': None,
                    'raw_response': text,
                    'parse_success': False
                })
                continue

            # Validate and normalize
            result = validate_evaluation_response(parsed, variant_type)
            result['#VariationID'] = custom_id
            result['raw_response'] = text
            result['parse_success'] = True

            results.append(result)

    df = pd.DataFrame(results)
    print(f"Parsed {len(df)} results from {results_path}")

    # Summary
    if 'parse_success' in df.columns:
        success_rate = df['parse_success'].mean() * 100
        print(f"Parse success rate: {success_rate:.1f}%")

    if 'concordance' in df.columns:
        print("\nConcordance distribution:")
        print(df['concordance'].value_counts())

    return df


def validate_evaluation_response(parsed: Dict[str, Any], variant_type: str = 'snp') -> Dict[str, Any]:
    """
    Validate and normalize the parsed JSON response.
    """
    expected_fields = {
        'concordance': 'PARSE_ERROR',
        'explanation': '',
        'signal_category': 'UNKNOWN',
        'primary_mechanism': '',
        'key_signals': '',
        'rationale_mechanism': '',
        'nt_missed': None,
        'notes': ''
    }

    if variant_type == 'indel':
        expected_fields['embedding_impact'] = 'UNKNOWN'

    valid_concordance = {'CONCORDANT', 'PARTIAL', 'DISCORDANT', 'NOT_APPLICABLE'}
    valid_signal_category = {'STRONG', 'MODERATE', 'WEAK', 'ABSENT', 'UNKNOWN'}

    result = {}
    for field, default in expected_fields.items():
        result[field] = parsed.get(field, default)

    if result['concordance'] not in valid_concordance:
        result['concordance'] = 'PARSE_ERROR'

    if result['signal_category'] not in valid_signal_category:
        result['signal_category'] = 'UNKNOWN'

    if result['nt_missed'] is not None:
        result['nt_missed'] = bool(result['nt_missed'])

    return result


# =============================================================================
# COMPLETE BATCH PIPELINE
# =============================================================================

def run_batch_evaluation_pipeline(
    df_with_prompts: pd.DataFrame,
    system_prompt: str,
    variant_type: str = 'snp',
    batch_name: str = 'variant_evaluation',
    work_dir: str = './batch_eval',
    api_key: str = None,
    model: str = "gemini-3-flash-preview",
    wait_for_completion: bool = True,
    poll_interval: int = 60
) -> Optional[pd.DataFrame]:
    """
    Run complete batch evaluation pipeline.

    Args:
        df_with_prompts: DataFrame with 'llm_prompt' column
        system_prompt: System instruction
        variant_type: 'snp' or 'indel'
        batch_name: Name for the batch job
        work_dir: Directory for intermediate files
        api_key: API key
        model: Model to use
        wait_for_completion: If True, wait and return results
        poll_interval: Seconds between status checks

    Returns:
        DataFrame with results if wait_for_completion, else None
    """
    # Create work directory
    work_path = Path(work_dir)
    work_path.mkdir(parents=True, exist_ok=True)

    timestamp = time.strftime('%Y%m%d_%H%M%S')

    # Paths
    jsonl_path = work_path / f'{batch_name}_{timestamp}_requests.jsonl'
    results_path = work_path / f'{batch_name}_{timestamp}_results.jsonl'
    job_info_path = work_path / f'{batch_name}_{timestamp}_job_info.json'

    # Step 1: Prepare JSONL
    print("=" * 60)
    print("STEP 1: Preparing batch request file")
    print("=" * 60)
    prepare_batch_jsonl(
        df_with_prompts=df_with_prompts,
        system_prompt=system_prompt,
        output_path=str(jsonl_path),
        variant_type=variant_type
    )

    # Step 2: Upload and submit
    print("\n" + "=" * 60)
    print("STEP 2: Uploading and submitting batch job")
    print("=" * 60)
    job_info = upload_and_submit_batch(
        jsonl_path=str(jsonl_path),
        display_name=f'{batch_name}_{timestamp}',
        api_key=api_key,
        model=model
    )

    # Save job info for later reference
    with open(job_info_path, 'w') as f:
        json.dump(job_info, f, indent=2)
    print(f"Job info saved to: {job_info_path}")

    if not wait_for_completion:
        print("\nBatch job submitted. Use check_batch_status() to monitor progress.")
        print(f"Job name: {job_info['job_name']}")
        return None

    # Step 3: Wait for completion
    print("\n" + "=" * 60)
    print("STEP 3: Waiting for batch completion")
    print("=" * 60)
    final_status = wait_for_batch_completion(
        job_name=job_info['job_name'],
        api_key=api_key,
        poll_interval=poll_interval
    )

    if final_status['state'] not in ['JobState.JOB_STATE_SUCCEEDED','JOB_STATE_SUCCEEDED', 'SUCCEEDED', 'STATE_SUCCEEDED']:
        print(f"\nBatch job did not complete successfully: {final_status['state']}")
        return None

    # Step 4: Download results
    print("\n" + "=" * 60)
    print("STEP 4: Downloading results")
    print("=" * 60)
    download_batch_results(
        job_name=job_info['job_name'],
        output_path=str(results_path),
        api_key=api_key
    )

    # Step 5: Parse results
    print("\n" + "=" * 60)
    print("STEP 5: Parsing results")
    print("=" * 60)
    results_df = parse_batch_results(
        results_path=str(results_path),
        variant_type=variant_type
    )

    # Save parsed results
    parquet_path = work_path / f'{batch_name}_{timestamp}_parsed.parquet'
    results_df.to_parquet(parquet_path)
    print(f"\nParsed results saved to: {parquet_path}")

    # Print summary
    print_evaluation_summary(results_df)

    return results_df


# =============================================================================
# UTILITY: Resume from job name
# =============================================================================

def resume_batch_evaluation(
    job_name: str,
    output_path: str,
    variant_type: str = 'snp',
    api_key: str = None,
    wait_if_pending: bool = True
) -> pd.DataFrame:
    """
    Resume/retrieve results from an existing batch job.

    Useful if you submitted a job and closed your session.

    Args:
        job_name: Name of the batch job
        output_path: Path to save results
        variant_type: 'snp' or 'indel'
        api_key: API key
        wait_if_pending: If True, wait for completion if job is still running

    Returns:
        DataFrame with results
    """
    # Check status
    status = check_batch_status(job_name, api_key)
    print(f"Job: {job_name}")
    print(f"State: {status['state']}\n")

    if status['state'] not in ['JOB_STATE_SUCCEEDED', 'SUCCEEDED', 'STATE_SUCCEEDED']:
        if wait_if_pending:
            print("Job not complete. Waiting...")
            status = wait_for_batch_completion(job_name, api_key)
        else:
            print("Job not complete yet.")
            return None

    # Download and parse
    results_jsonl = output_path.replace('.parquet', '_raw.jsonl')
    download_batch_results(job_name, results_jsonl, api_key)

    results_df = parse_batch_results(results_jsonl, variant_type)
    results_df.to_parquet(output_path)

    return results_df


# =============================================================================
# SUMMARY FUNCTIONS
# =============================================================================

def print_evaluation_summary(results_df: pd.DataFrame) -> None:
    """Print formatted summary of evaluation results."""
    total = len(results_df)

    print("\n" + "=" * 60)
    print("EVALUATION SUMMARY")
    print("=" * 60)
    print(f"Total variants evaluated: {total}")

    if 'parse_success' in results_df.columns:
        success_rate = results_df['parse_success'].mean() * 100
        print(f"Parse success rate: {success_rate:.1f}%")

    print("\nConcordance Distribution:")
    print("-" * 40)

    concordance_counts = results_df['concordance'].value_counts()
    for cat in ['CONCORDANT', 'PARTIAL', 'DISCORDANT', 'NOT_APPLICABLE', 'PARSE_ERROR', 'API_ERROR']:
        count = concordance_counts.get(cat, 0)
        pct = count / total * 100 if total > 0 else 0
        bar = '█' * int(pct / 2)
        print(f"  {cat:16} {count:5} ({pct:5.1f}%) {bar}")

    if 'signal_category' in results_df.columns:
        print("\nSignal Category Distribution:")
        print("-" * 40)
        for cat, count in results_df['signal_category'].value_counts().items():
            pct = count / total * 100
            print(f"  {cat:12} {count:5} ({pct:5.1f}%)")

    if 'nt_missed' in results_df.columns:
        missed_rate = results_df['nt_missed'].mean()
        if not pd.isna(missed_rate):
            print(f"\nNT Missed Rate: {missed_rate*100:.1f}%")

    print("=" * 60)


# =============================================================================
# EXAMPLE USAGE
# =============================================================================

if __name__ == "__main__":
    import pyarrow.parquet as pq

    # Configuration reminder
    # PATHS = {
    #     "snp_annotated_signaled": "parquet/annotated_snps_signaled.parquet",
    #     "indel_annotated_signaled": "parquet/annotated_indels_signaled.parquet",
    # }

    API_KEY = "REDACTED_API_KEY"
    BATCH_MODEL = "gemini-3-flash-preview"

    # --- SNP Evaluation ---
    print("Loading SNP data...")
    snp_df = pq.read_table(PATHS["snp_annotated_signaled"]).to_pandas()

    # Optionally sample for testing
    # snp_df = snp_df.sample(n=10, random_state=42)

    print("Building prompts...")
    snp_with_prompts = build_prompts_table(snp_df, variant_type='snp')

    system_prompt = get_system_prompt(variant_type='snp')

    # Run batch evaluation
    results = run_batch_evaluation_pipeline(
        df_with_prompts=snp_with_prompts,
        system_prompt=system_prompt,
        variant_type='snp',
        batch_name='snp_concordance_eval',
        work_dir='./batch_eval/snps',
        api_key=API_KEY,
        model="gemini-3-flash-preview",
        wait_for_completion=True,
        poll_interval=60
    )




# --- merge back to the vairant df

    # if results is not None:
    #     # Merge back to original data
    #     snp_with_eval = snp_df.merge(
    #         results[['#VariationID', 'concordance', 'explanation', 'signal_category',
    #                  'primary_mechanism', 'nt_missed']],
    #         on='#VariationID',
    #         how='left'
    #     )
    #     snp_with_eval.to_parquet('parquet/snps_with_evaluation.parquet')
    #     print("Merged results saved!")

    # --- for resuming processing of an already sent batc
    # results = resume_batch_evaluation(
    #     job_name='batches/your-job-id',
    #     output_path='batch_eval/snps/resumed_results.parquet',
    #     variant_type='snp',
    #     api_key=API_KEY
    # )




Loading SNP data...
Building prompts...
STEP 1: Preparing batch request file
Created batch file: batch_eval\snps\snp_concordance_eval_20260214_100338_requests.jsonl
Total requests: 10

STEP 2: Uploading and submitting batch job
Uploading batch_eval\snps\snp_concordance_eval_20260214_100338_requests.jsonl...
Uploaded: files/c3qboo6skn7n
Creating batch job...
Batch job created: batches/56olxibmisbmcz41b5c20sx1pcpu2qqrx1oc
Status: JobState.JOB_STATE_PENDING
Job info saved to: batch_eval\snps\snp_concordance_eval_20260214_100338_job_info.json

STEP 3: Waiting for batch completion
Waiting for batch job: batches/56olxibmisbmcz41b5c20sx1pcpu2qqrx1oc
Polling every 60s, timeout 7200s
JobState.JOB_STATE_PENDING
[0.0min] State: JobState.JOB_STATE_PENDINGJobState.JOB_STATE_PENDING
[1.1min] State: JobState.JOB_STATE_PENDINGJobState.JOB_STATE_PENDING
[2.1min] State: JobState.JOB_STATE_PENDINGJobState.JOB_STATE_PENDING
[3.1min] State: JobState.JOB_STATE_PENDINGJobState.JOB_STATE_PENDING
[4.1min] Stat